In [21]:
# [CELL 1] - Kaggle dependencies for local Devstral Small 2 via vLLM
!pip install -q -U "vllm" "mistral-common>=1.8.6" "openai" "huggingface_hub" "easyocr" "scikit-image" "opencv-python-headless" "ipywidgets" "nest_asyncio" "pillow"
import os
import nest_asyncio
os.environ.setdefault("HF_HOME", "/kaggle/working/hf-cache")
os.environ.setdefault("VLLM_WORKER_MULTIPROC_METHOD", "spawn")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
nest_asyncio.apply()
print("✅ Dependencies installed")
print("✅ Model: mistralai/Devstral-Small-2-24B-Instruct-2512")
print("✅ Backend: local vLLM · 2×T4")


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydata-profiling 4.18.4 requires numba<0.63,>=0.60, but you have numba 0.65.0 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
cucim-cu12 26.2.0 requires scikit-image<0.26.0,>=0.19.0, but you have scikit-image 0.26.0 which is incompatible.
gradio 5.50.0 requires pillow<12.0,>=8.0, but you have pillow 12.3.0 which is incompatible.
gradio 5.50.0 requires starlette<1.0,>=0.40.0, but you have starlette 1.6.0 which is incompatible.
✅ Dependencies installed
✅ Model: mistralai/Devstral-Small-2-24B-Instruct-2512
✅ Backend: local vLLM · 2×T4


In [22]:
# ============================================================
# DEVSTRAL KAGGLE ENVIRONMENT CHECK
# 2× NVIDIA T4 | ~30 GiB RAM | ~19.5 GiB free storage
# ============================================================

import sys
import subprocess
import shutil

# 1. Install only missing package
try:
    import bitsandbytes
    print(f"✅ bitsandbytes already installed: {bitsandbytes.__version__}")
except ImportError:
    print("📦 Installing bitsandbytes...")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "bitsandbytes"
    ])
    import bitsandbytes
    print(f"✅ bitsandbytes installed: {bitsandbytes.__version__}")

# 2. Core packages
import torch
import transformers
import accelerate

# 3. Storage
total, used, free = shutil.disk_usage("/kaggle/working")

# 4. System information
print("\n" + "=" * 60)
print("🧪 KAGGLE DEVSTRAL ENVIRONMENT")
print("=" * 60)

print(f"Python:        {sys.version.split()[0]}")
print(f"PyTorch:       {torch.__version__}")
print(f"Transformers:  {transformers.__version__}")
print(f"Accelerate:    {accelerate.__version__}")
print(f"BitsAndBytes:  {bitsandbytes.__version__}")
print(f"CUDA:          {torch.version.cuda}")
print(f"CUDA available:{torch.cuda.is_available()}")
print(f"GPU count:     {torch.cuda.device_count()}")

print("\n🖥️ GPUs:")
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    total_vram = props.total_memory / 1024**3
    free_vram, used_vram = torch.cuda.mem_get_info(i)
    
    print(
        f"  GPU {i}: {props.name} | "
        f"Total: {total_vram:.2f} GiB | "
        f"Free: {free_vram/1024**3:.2f} GiB"
    )

print("\n💾 Storage:")
print(f"  Total: {total/1024**3:.2f} GiB")
print(f"  Used:  {used/1024**3:.2f} GiB")
print(f"  Free:  {free/1024**3:.2f} GiB")

print("\n" + "=" * 60)

# 5. CUDA sanity test
if torch.cuda.is_available():
    try:
        for i in range(torch.cuda.device_count()):
            x = torch.randn(1024, 1024, device=f"cuda:{i}")
            y = x @ x
            torch.cuda.synchronize(i)
            del x, y
            torch.cuda.empty_cache()

        print("✅ CUDA GPU computation test: PASSED")
    except Exception as e:
        print(f"❌ CUDA test failed: {e}")
else:
    print("❌ CUDA is not available")

# 6. Final recommendation
print("\n📌 STATUS")
if (
    torch.cuda.device_count() >= 2
    and torch.cuda.is_available()
    and free / 1024**3 > 15
):
    print("✅ Hardware is ready for a small quantized Devstral setup.")
    print("⚠️ Do NOT download the original 51+ GB Devstral model.")
    print("➡️ Next step: use a ≤15 GB quantized Devstral build.")
else:
    print("⚠️ Hardware/storage needs attention before model download.")

print("=" * 60)

✅ bitsandbytes already installed: 0.50.2

🧪 KAGGLE DEVSTRAL ENVIRONMENT
Python:        3.12.13
PyTorch:       2.13.0+cu130
Transformers:  5.17.0
Accelerate:    1.13.0
BitsAndBytes:  0.50.2
CUDA:          13.0
CUDA available:True
GPU count:     2

🖥️ GPUs:
  GPU 0: Tesla T4 | Total: 14.56 GiB | Free: 14.43 GiB
  GPU 1: Tesla T4 | Total: 14.56 GiB | Free: 14.43 GiB

💾 Storage:
  Total: 19.52 GiB
  Used:  0.00 GiB
  Free:  19.50 GiB

✅ CUDA GPU computation test: PASSED

📌 STATUS
✅ Hardware is ready for a small quantized Devstral setup.
⚠️ Do NOT download the original 51+ GB Devstral model.
➡️ Next step: use a ≤15 GB quantized Devstral build.


In [23]:
# [CELL 2] - Devstral local backend diagnostic
import torch, shutil
print("torch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPUs:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))
print("vLLM:", shutil.which("vllm") or "NOT FOUND")
print("Model:", "mistralai/Devstral-Small-2-24B-Instruct-2512")


torch: 2.13.0+cu130
CUDA: 13.0
GPUs: 2
0 Tesla T4
1 Tesla T4
vLLM: /usr/local/bin/vllm
Model: mistralai/Devstral-Small-2-24B-Instruct-2512


In [24]:
# ==============================================================
# 🔧 CLEAN FIX: PILLOW / EASYOCR
# ==============================================================

import os
import sys
import glob
import shutil
import subprocess

print("=" * 70)
print("🔧 CLEANING PILLOW / EASYOCR")
print("=" * 70)

# --------------------------------------------------------------
# 1. Uninstall Pillow
# --------------------------------------------------------------
print("\n[1/5] Uninstalling Pillow...")
subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "-y", "Pillow", "PIL"],
    check=False
)

# --------------------------------------------------------------
# 2. Remove ALL leftover PIL/Pillow files
# --------------------------------------------------------------
print("[2/5] Removing leftover PIL/Pillow files...")

site_packages = "/usr/local/lib/python3.12/dist-packages"

for path in glob.glob(os.path.join(site_packages, "PIL")):
    print("Removing:", path)
    shutil.rmtree(path, ignore_errors=True)

for pattern in [
    "Pillow-*.dist-info",
    "Pillow-*.egg-info",
    "PIL-*.dist-info",
    "PIL-*.egg-info",
]:
    for path in glob.glob(os.path.join(site_packages, pattern)):
        print("Removing:", path)
        shutil.rmtree(path, ignore_errors=True)

# --------------------------------------------------------------
# 3. Install a clean Pillow 12.x
# --------------------------------------------------------------
print("\n[3/5] Installing clean Pillow 12.0.0...")
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--no-cache-dir",
        "--force-reinstall",
        "--no-deps",
        "Pillow==12.0.0",
    ],
    check=True
)

print("✅ Pillow 12.0.0 installed.")

# --------------------------------------------------------------
# 4. IMPORTANT: restart Kaggle session
# --------------------------------------------------------------
print("\n[4/5] IMPORTANT")
print("⚠️  DO NOT continue running the notebook from this cell.")
print("⚠️  Restart the Kaggle session/kernel now.")
print("")
print("Kaggle:")
print("  Session → Restart Session")
print("")
print("After restart, run the next test cell.")

🔧 CLEANING PILLOW / EASYOCR

[1/5] Uninstalling Pillow...
Found existing installation: pillow 12.3.0
Uninstalling pillow-12.3.0:
  Successfully uninstalled pillow-12.3.0


[2/5] Removing leftover PIL/Pillow files...

[3/5] Installing clean Pillow 12.0.0...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 40.6 MB/s eta 0:00:00
✅ Pillow 12.0.0 installed.

[4/5] IMPORTANT
⚠️  DO NOT continue running the notebook from this cell.
⚠️  Restart the Kaggle session/kernel now.

Kaggle:
  Session → Restart Session

After restart, run the next test cell.


In [25]:
# [CELL 3] - Kaggle Configuration
KAGGLE_BASE = "/kaggle/working/ui_converter"

CONFIG = {
    "input_source": "upload",
    "output_path": f"{KAGGLE_BASE}/output/",
    "input_path": f"{KAGGLE_BASE}/input/",
    "ocr_enabled": True,
    "ui_detection_enabled": True,
    "generate_html": True,
    "generate_react": True,
    "generate_tailwind": True,
    "generate_figma": True,
    "responsive": True,
    "pixel_accuracy": True,
    "ocr_lang": ["en"],
    "max_refinement_iterations": 2,
    "target_similarity": 0.90,
    "ai_engine": "devstral_24b",
    "local_only": True,
}

os.makedirs(CONFIG["output_path"], exist_ok=True)
os.makedirs(CONFIG["input_path"], exist_ok=True)

print("✅ Kaggle configuration loaded.")
print("   Output:", CONFIG["output_path"])
print("   Local LLM:", "mistralai/Devstral-Small-2-24B-Instruct-2512")


✅ Kaggle configuration loaded.
   Output: /kaggle/working/ui_converter/output/
   Local LLM: mistralai/Devstral-Small-2-24B-Instruct-2512


In [26]:
# [CELL 4] - Kaggle Input Utilities

import os
import cv2
from typing import Dict, Any


class InputManager:
    def __init__(self, config: Dict[str, Any]):
        self.config = config

        self.input_dir = config.get(
            "input_path",
            "/kaggle/working/ui_converter/input/"
        )

        self.supported_formats = (
            ".png",
            ".jpg",
            ".jpeg",
            ".webp"
        )

        os.makedirs(self.input_dir, exist_ok=True)

    def mount_drive(self):
        raise RuntimeError(
            "Google Drive mounting is not used in the Kaggle version. "
            "Upload screenshots with the Upload Image(s) widget."
        )

    def fetch_from_drive(self):
        return []

    def handle_upload(self, uploaded_data):
        files_to_process = []

        if isinstance(uploaded_data, (tuple, list)):
            items = uploaded_data

        elif isinstance(uploaded_data, dict):
            items = []

            for filename, data in uploaded_data.items():
                if isinstance(data, dict):
                    item = dict(data)
                    item.setdefault("name", filename)
                    items.append(item)
                else:
                    items.append({
                        "name": filename,
                        "content": data,
                    })
        else:
            items = []

        for item in items:
            filename = str(item.get("name", "")).strip()
            content = item.get("content")

            if (
                filename.lower().endswith(self.supported_formats)
                and content is not None
            ):
                safe_name = os.path.basename(filename)
                path = os.path.join(self.input_dir, safe_name)

                # Handle widget data that may be memoryview/bytearray
                if isinstance(content, memoryview):
                    content = content.tobytes()
                elif isinstance(content, bytearray):
                    content = bytes(content)

                with open(path, "wb") as f:
                    f.write(content)

                files_to_process.append(path)

        logger.info(
            "Uploaded %d image(s).",
            len(files_to_process)
        )

        return files_to_process


class ImageProcessor:
    @staticmethod
    def preprocess(image_path: str, output_dir: str) -> Dict[str, Any]:
        """Normalize image, enhance contrast, and save for processing."""

        filename = os.path.basename(image_path)

        img = cv2.imread(image_path)

        if img is None:
            raise ValueError(f"Invalid image file: {image_path}")

        height, width = img.shape[:2]

        # Grayscale
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        # Contrast enhancement
        clahe = cv2.createCLAHE(
            clipLimit=2.0,
            tileGridSize=(8, 8)
        )

        enhanced_gray = clahe.apply(gray)

        # Edge detection for UI boundaries
        edges = cv2.Canny(
            enhanced_gray,
            50,
            150
        )

        # Save processed image
        proc_dir = os.path.join(
            output_dir,
            "processed"
        )

        os.makedirs(proc_dir, exist_ok=True)

        proc_path = os.path.join(
            proc_dir,
            f"enhanced_{filename}"
        )

        cv2.imwrite(
            proc_path,
            enhanced_gray
        )

        return {
            "original_path": image_path,
            "processed_path": proc_path,
            "edges": edges,
            "image": img,
            "dimensions": {
                "width": width,
                "height": height
            }
        }


print("✅ Cell 4 loaded successfully.")

✅ Cell 4 loaded successfully.


In [27]:
# [CELL 5] - Analysis Engines

import os
import uuid
import logging
import cv2
import numpy as np
import easyocr

from typing import List, Dict, Any, Optional


# --------------------------------------------------------------
# Logger fallback
# --------------------------------------------------------------
logger = logging.getLogger(__name__)

if not logger.handlers:
    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s | %(levelname)s | %(message)s"
    )


# ==============================================================
# OCR PROCESSOR
# ==============================================================
class OCRProcessor:
    def __init__(self, langs: Optional[List[str]] = None):
        if langs is None:
            langs = ["en"]

        logger.info(
            "Initializing EasyOCR on CPU "
            "(GPU memory is reserved for Devstral)..."
        )

        # Keep EasyOCR on CPU so T4 VRAM remains available for Devstral
        self.reader = easyocr.Reader(
            langs,
            gpu=False,
            verbose=False
        )

    def extract_text(self, image_path: str) -> List[Dict[str, Any]]:
        if not os.path.isfile(image_path):
            raise FileNotFoundError(
                f"Image not found: {image_path}"
            )

        results = self.reader.readtext(image_path)

        text_elements = []

        for bbox, text, prob in results:
            if float(prob) <= 0.3:
                continue

            # bbox:
            # [
            #   [x1, y1],
            #   [x2, y2],
            #   [x3, y3],
            #   [x4, y4]
            # ]

            x_coords = [p[0] for p in bbox]
            y_coords = [p[1] for p in bbox]

            x = int(min(x_coords))
            y = int(min(y_coords))

            w = int(max(x_coords) - x)
            h = int(max(y_coords) - y)

            text_elements.append({
                "id": f"text_{uuid.uuid4().hex[:8]}",
                "type": "text",
                "text": str(text),
                "x": x,
                "y": y,
                "width": w,
                "height": h,
                "confidence": float(prob)
            })

        logger.info(
            "OCR extracted %d text element(s) from %s",
            len(text_elements),
            os.path.basename(image_path)
        )

        return text_elements


# ==============================================================
# UI ELEMENT DETECTOR
# ==============================================================
class UIElementDetector:

    @staticmethod
    def detect_elements(
        preprocessed_data: Dict[str, Any],
        ocr_data: List[Dict[str, Any]]
    ) -> List[Dict[str, Any]]:
        """
        Uses contour detection + OCR data to infer UI components.
        """

        edges = preprocessed_data["edges"]
        img = preprocessed_data["image"]

        if edges is None:
            raise ValueError("Preprocessed edge image is missing.")

        if img is None:
            raise ValueError("Original image is missing.")

        # ------------------------------------------------------
        # Detect contours
        # ------------------------------------------------------
        contours, hierarchy = cv2.findContours(
            edges,
            cv2.RETR_EXTERNAL,
            cv2.CHAIN_APPROX_SIMPLE
        )

        elements = []

        for cnt in contours:
            x, y, w, h = cv2.boundingRect(cnt)
            area = w * h

            # --------------------------------------------------
            # Filter noise
            # --------------------------------------------------
            if area < 400 or w < 20 or h < 20:
                continue

            # --------------------------------------------------
            # Calculate average/background color
            # --------------------------------------------------
            mask = np.zeros(
                img.shape[:2],
                dtype=np.uint8
            )

            cv2.drawContours(
                mask,
                [cnt],
                -1,
                255,
                -1
            )

            mean_val = cv2.mean(
                img,
                mask=mask
            )[:3]

            # BGR -> RGB
            hex_color = (
                "#{:02x}{:02x}{:02x}".format(
                    int(mean_val[2]),
                    int(mean_val[1]),
                    int(mean_val[0])
                )
            )

            # --------------------------------------------------
            # Determine UI element type
            # --------------------------------------------------
            aspect_ratio = w / max(h, 1)

            el_type = "container"

            if aspect_ratio > 3:
                el_type = (
                    "divider"
                    if h < 10
                    else "input"
                )

            elif 0.8 < aspect_ratio < 1.2:
                el_type = (
                    "image_placeholder"
                    if area > 2500
                    else "icon"
                )

            # --------------------------------------------------
            # Detect text contained within this element
            # --------------------------------------------------
            contained_text = []

            for txt in ocr_data:
                tx = txt["x"]
                ty = txt["y"]
                tw = txt["width"]
                th = txt["height"]

                text_inside = (
                    x <= tx
                    and y <= ty
                    and (x + w) >= (tx + tw)
                    and (y + h) >= (ty + th)
                )

                if text_inside:
                    contained_text.append(txt)

            # --------------------------------------------------
            # Refine type using OCR
            # --------------------------------------------------
            if contained_text and el_type == "container" and h < 60:
                el_type = "button"

            elif contained_text and area > 10000:
                el_type = "card"

            # --------------------------------------------------
            # Add detected element
            # --------------------------------------------------
            elements.append({
                "id": f"el_{uuid.uuid4().hex[:8]}",
                "type": el_type,
                "x": int(x),
                "y": int(y),
                "width": int(w),
                "height": int(h),
                "background": hex_color,
                "children": [
                    t["text"]
                    for t in contained_text
                ]
            })

        # ------------------------------------------------------
        # Add OCR text as standalone elements
        # ------------------------------------------------------
        elements.extend(ocr_data)

        # ------------------------------------------------------
        # Sort top-to-bottom, left-to-right
        # ------------------------------------------------------
        elements.sort(
            key=lambda e: (
                e.get("y", 0),
                e.get("x", 0)
            )
        )

        logger.info(
            "Detected %d UI element(s)",
            len(elements)
        )

        return elements


# ==============================================================
# CELL TEST
# ==============================================================
print("✅ Cell 5 loaded successfully.")
print("✅ OCRProcessor available")
print("✅ UIElementDetector available")

✅ Cell 5 loaded successfully.
✅ OCRProcessor available
✅ UIElementDetector available


In [28]:
# [CELL 6] - Architecture & Validation
class UIJSONBuilder:
    @staticmethod
    def build(dimensions: Dict, elements: List[Dict], filename: str) -> Dict:
        return {
            "page": {
                "name": os.path.splitext(filename)[0],
                "width": dimensions["width"],
                "height": dimensions["height"]
            },
            "tokens": {
                "colors": list(set([e.get("background") for e in elements if e.get("background")])),
            },
            "elements": elements
        }

class Validator:
    @staticmethod
    def validate_visual(original_img: np.ndarray, generated_html_path: str) -> float:
        """
        Calculates SSIM. (In a full env, this would render HTML headless first.
        Here we mock the render comparison for Colab stability).
        """
        # Mocking similarity score based on element bounding box coverage
        # True pixel validation requires Puppeteer/Selenium which is unstable in Colab
        return round(np.random.uniform(0.85, 0.95), 2) # Simulated SSIM


In [29]:
# ============================================================
# [CELL 07] - Local Devstral Small 2 24B Multimodal Backend
# ============================================================
import os
import re
import json
import time
import base64
import mimetypes
import shutil
import subprocess
import urllib.request
from pathlib import Path

import torch
from openai import OpenAI

MODEL_ID = "mistralai/Devstral-Small-2-24B-Instruct-2512"
DEVSTRAL_MODEL_ID = MODEL_ID
DEVSTRAL_HOST = "127.0.0.1"
DEVSTRAL_PORT = 8000
DEVSTRAL_BASE_URL = f"http://{DEVSTRAL_HOST}:{DEVSTRAL_PORT}/v1"
DEVSTRAL_HEALTH_URL = f"http://{DEVSTRAL_HOST}:{DEVSTRAL_PORT}/v1/models"
DEVSTRAL_LOG_PATH = "/kaggle/working/devstral-vllm.log"
DEVSTRAL_START_TIMEOUT = 1800
DEVSTRAL_MAX_NEW_TOKENS = 4096
DEVSTRAL_CONTEXT = 8192

devstral_server_process = None


def _server_payload():
    try:
        with urllib.request.urlopen(DEVSTRAL_HEALTH_URL, timeout=4) as response:
            return json.loads(response.read().decode("utf-8"))
    except Exception:
        return None


def _devstral_server_ready():
    data = _server_payload()
    if not data:
        return False
    return any(x.get("id") == MODEL_ID for x in data.get("data", []))


def _tail_log(lines=80):
    p = Path(DEVSTRAL_LOG_PATH)
    if not p.exists():
        return "<server log unavailable>"
    return "\n".join(p.read_text(encoding="utf-8", errors="replace").splitlines()[-lines:])


def _stop_previous_server():
    try:
        subprocess.run(
            ["bash", "-lc", "pkill -f 'vllm serve mistralai/Devstral-Small-2-24B-Instruct-2512' || true"],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
            timeout=10,
        )
    except Exception:
        pass


def load_devstral_model(force_restart=False):
    global devstral_server_process

    if _devstral_server_ready() and not force_restart:
        return

    if force_restart:
        _stop_previous_server()
        time.sleep(2)

    vllm_bin = shutil.which("vllm")
    if not vllm_bin:
        raise RuntimeError("vLLM was not found. Re-run Cell 1.")

    gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 0
    gpu_names = [torch.cuda.get_device_name(i) for i in range(gpu_count)]
    if gpu_count < 2:
        raise RuntimeError(
            f"This local Devstral notebook requires 2 GPUs. Found {gpu_count}: {gpu_names}."
        )

    Path(DEVSTRAL_LOG_PATH).parent.mkdir(parents=True, exist_ok=True)

    # Devstral Small 2 is an FP8 checkpoint. vLLM supports running FP8 models
    # on Turing GPUs such as T4 through its weight-only W8A16 path.
    cmd = [
        vllm_bin, "serve", MODEL_ID,
        "--host", DEVSTRAL_HOST,
        "--port", str(DEVSTRAL_PORT),
        "--tensor-parallel-size", "2",
        "--tokenizer-mode", "mistral",
        "--max-model-len", str(DEVSTRAL_CONTEXT),
        "--max-num-seqs", "1",
        "--max-num-batched-tokens", "4096",
        "--gpu-memory-utilization", "0.90",
        "--served-model-name", MODEL_ID,
        "--enforce-eager",
        "--disable-log-requests",
    ]

    print("CUDA devices:", gpu_count, gpu_names)
    print("🚀 Starting local Devstral Small 2...")
    print("   Model:", MODEL_ID)
    print("   Tensor parallel: 2 GPUs")
    print("   Context:", DEVSTRAL_CONTEXT)
    print("   Log:", DEVSTRAL_LOG_PATH)

    with open(DEVSTRAL_LOG_PATH, "w", encoding="utf-8", buffering=1) as log_file:
        devstral_server_process = subprocess.Popen(
            cmd,
            stdout=log_file,
            stderr=subprocess.STDOUT,
            start_new_session=True,
            env=os.environ.copy(),
        )

    deadline = time.time() + DEVSTRAL_START_TIMEOUT
    last_report = 0
    while time.time() < deadline:
        if _devstral_server_ready():
            print("✅ Devstral server READY")
            return

        if devstral_server_process.poll() is not None:
            raise RuntimeError(
                "Devstral vLLM server exited during startup.\n\n" + _tail_log()
            )

        now = time.time()
        if now - last_report >= 15:
            print("⏳ Waiting for model download/load and vLLM startup...")
            last_report = now
        time.sleep(3)

    raise TimeoutError(
        "Timed out waiting for Devstral vLLM server.\n\n" + _tail_log()
    )


def _devstral_client():
    load_devstral_model()
    return OpenAI(base_url=DEVSTRAL_BASE_URL, api_key="local", timeout=1800.0)


def _image_data_uri(image_path):
    path = Path(image_path)
    if not path.is_file():
        raise FileNotFoundError(f"Image not found: {path}")
    mime = mimetypes.guess_type(path.name)[0] or "image/png"
    encoded = base64.b64encode(path.read_bytes()).decode("ascii")
    return f"data:{mime};base64,{encoded}"


def query_devstral(prompt, image_path=None, max_new_tokens=DEVSTRAL_MAX_NEW_TOKENS, temperature=0.15):
    if not prompt or not prompt.strip():
        raise ValueError("Prompt cannot be empty.")

    client = _devstral_client()
    content = prompt
    if image_path:
        content = [
            {"type": "text", "text": prompt},
            {"type": "image_url", "image_url": {"url": _image_data_uri(image_path)}},
        ]

    response = client.chat.completions.create(
        model=MODEL_ID,
        messages=[
            {"role": "system", "content": "You are Devstral, a precise software-engineering and UI reconstruction model."},
            {"role": "user", "content": content},
        ],
        temperature=temperature,
        max_tokens=max_new_tokens,
    )
    text = response.choices[0].message.content
    if not text or not text.strip():
        raise RuntimeError("Devstral returned an empty response.")
    return text.strip()


def query_devstral_vision(image_path, prompt):
    return query_devstral(prompt=prompt, image_path=image_path)


def query_devstral_text(prompt):
    return query_devstral(prompt=prompt)


def query_vision_model(image_path, prompt, engine="devstral_24b"):
    if str(engine).strip().lower() != "devstral_24b":
        raise ValueError("This notebook is local-only; use Devstral 24B.")
    logger.info("Vision request -> devstral_24b")
    return query_devstral_vision(image_path, prompt)


def query_text_model(prompt, engine="devstral_24b"):
    if str(engine).strip().lower() != "devstral_24b":
        raise ValueError("This notebook is local-only; use Devstral 24B.")
    logger.info("Text request -> devstral_24b")
    return query_devstral_text(prompt)


def clean_code_block(raw_text, language):
    if not raw_text:
        return ""
    text = raw_text.strip()
    aliases = {"html": r"html?", "jsx": r"(?:jsx|tsx|javascript|react)"}
    lang = aliases.get(language, re.escape(language))
    m = re.search(rf"```\s*{lang}\s*([\s\S]*?)\s*```", text, re.IGNORECASE)
    if m:
        return m.group(1).strip()
    m = re.search(r"```\s*([\s\S]*?)\s*```", text)
    return m.group(1).strip() if m else text


class HTMLGenerator:
    @staticmethod
    def generate(image_path, out_dir, engine="devstral_24b"):
        html_dir = Path(out_dir) / "html"
        html_dir.mkdir(parents=True, exist_ok=True)
        prompt = """
You are a principal frontend engineer specializing in screenshot-to-code reconstruction.
Analyze the supplied screenshot and create a highly accurate responsive HTML5 implementation.

Return ONLY the complete HTML document.

Requirements:
- Start with <!DOCTYPE html>.
- Match geometry, spacing, colors, typography, borders, shadows, radii and alignment closely.
- Reproduce readable text exactly.
- Use semantic HTML and accessible controls.
- Put CSS in the document.
- Use inline SVG/CSS/simple placeholders instead of inventing external asset URLs.
- Make it standalone and responsive.
- Do not use Markdown fences or explanations.
"""
        logger.info("Generating HTML via [DEVSTRAL_24B]...")
        raw = query_vision_model(image_path, prompt, engine)
        code = clean_code_block(raw, "html")
        if "<html" not in code.lower() and "<!doctype" not in code.lower():
            raise RuntimeError("Devstral did not return a complete HTML document.")
        path = html_dir / "index.html"
        path.write_text(code, encoding="utf-8")
        return str(path)


class ReactTailwindGenerator:
    @staticmethod
    def generate(image_path, out_dir, engine="devstral_24b"):
        root = Path(out_dir) / "react"
        src = root / "src"
        src.mkdir(parents=True, exist_ok=True)
        prompt = """
You are an expert React frontend engineer. Recreate the supplied screenshot as a pixel-accurate responsive React + Tailwind UI.

Return ONLY App.jsx. No Markdown fences. No explanation.

Requirements:
- Use export default function App() { ... }.
- Use semantic, accessible JSX.
- Match layout, dimensions, spacing, typography, colors, borders, shadows and radii.
- Reproduce readable text exactly.
- Do not rely on invented external image URLs for core visuals.
- Keep the component self-contained and Vite-compatible.
"""
        logger.info("Generating React via [DEVSTRAL_24B]...")
        raw = query_vision_model(image_path, prompt, engine)
        jsx = clean_code_block(raw, "jsx")
        if "function App" not in jsx and "export default" not in jsx:
            raise RuntimeError("Devstral did not return a recognizable React App component.")

        (src / "App.jsx").write_text(jsx, encoding="utf-8")
        (root / "package.json").write_text(json.dumps({
            "name": "screenshot-to-ui-react",
            "private": True,
            "version": "1.0.0",
            "type": "module",
            "scripts": {"dev": "vite", "build": "vite build", "preview": "vite preview"},
            "dependencies": {"react": "^18.3.1", "react-dom": "^18.3.1"},
            "devDependencies": {"@vitejs/plugin-react": "^4.3.1", "tailwindcss": "^3.4.1", "vite": "^5.4.2"}
        }, indent=2), encoding="utf-8")
        (root / "vite.config.js").write_text(
            "import { defineConfig } from 'vite'\nimport react from '@vitejs/plugin-react'\nexport default defineConfig({plugins:[react()]})\n",
            encoding="utf-8"
        )
        (root / "index.html").write_text(
            "<!doctype html><html lang='en'><head><meta charset='UTF-8'><meta name='viewport' content='width=device-width,initial-scale=1.0'><title>Screenshot UI</title></head><body><div id='root'></div><script type='module' src='/src/main.jsx'></script></body></html>",
            encoding="utf-8"
        )
        (src / "main.jsx").write_text(
            "import React from 'react';\nimport ReactDOM from 'react-dom/client';\nimport App from './App';\nimport './index.css';\nReactDOM.createRoot(document.getElementById('root')).render(<React.StrictMode><App /></React.StrictMode>);\n",
            encoding="utf-8"
        )
        (src / "index.css").write_text(
            "@tailwind base;\n@tailwind components;\n@tailwind utilities;\nhtml,body,#root{min-height:100%;}body{margin:0;}\n",
            encoding="utf-8"
        )
        (root / "tailwind.config.js").write_text(
            "export default {content:['./index.html','./src/**/*.{js,jsx,ts,tsx}'],theme:{extend:{}},plugins:[]};\n",
            encoding="utf-8"
        )
        return str(root)


print("✅ Local Devstral 24B router/generators loaded")


✅ Local Devstral 24B router/generators loaded


In [35]:
# [CELL 8] - Figma Generator (Plugin Architecture as requested)
class FigmaGenerator:
    @staticmethod
    def generate_plugin(ui_json: Dict, out_dir: str):
        """Generates a fully functional Figma Plugin to import the UI JSON."""
        figma_dir = os.path.join(out_dir, "figma_plugin")
        os.makedirs(figma_dir, exist_ok=True)

        # 1. Manifest
        manifest = {
            "name": f"Import {ui_json['page']['name']}",
            "id": "ui-converter-plugin",
            "api": "1.0.0",
            "main": "code.js",
            "ui": "ui.html"
        }
        with open(os.path.join(figma_dir, "manifest.json"), "w") as f:
            json.dump(manifest, f, indent=2)

        # 2. ui.json payload wrapper
        with open(os.path.join(figma_dir, "ui.json"), "w") as f:
            json.dump(ui_json, f, indent=2)

        # 3. code.js (Figma API interactions)
        code_js = """
        figma.showUI(__html__, { width: 300, height: 200 });
        figma.ui.onmessage = async msg => {
            if (msg.type === 'generate') {
                const data = msg.data;
                const frame = figma.createFrame();
                frame.name = data.page.name;
                frame.resize(data.page.width, data.page.height);

                // Load font first
                await figma.loadFontAsync({ family: "Inter", style: "Regular" });

                for (const el of data.elements) {
                    if (el.type === 'text') {
                        const text = figma.createText();
                        text.x = el.x; text.y = el.y;
                        text.characters = el.text;
                        frame.appendChild(text);
                    } else {
                        const rect = figma.createRectangle();
                        rect.x = el.x; rect.y = el.y;
                        rect.resize(el.width, el.height);

                        // Parse Hex to RGB
                        if(el.background && el.background.startsWith('#')) {
                            const hex = el.background.replace('#', '');
                            const r = parseInt(hex.substring(0, 2), 16) / 255;
                            const g = parseInt(hex.substring(2, 4), 16) / 255;
                            const b = parseInt(hex.substring(4, 6), 16) / 255;
                            rect.fills = [{type: 'SOLID', color: {r, g, b}}];
                        }
                        if (el.type === 'button') rect.cornerRadius = 8;
                        frame.appendChild(rect);
                    }
                }
                figma.viewport.scrollAndZoomIntoView([frame]);
                figma.closePlugin();
            }
        };
        """
        with open(os.path.join(figma_dir, "code.js"), "w") as f:
            f.write(code_js)

        # 4. ui.html
        ui_html = f"""
        <h2>Figma UI Builder</h2>
        <p>Ready to generate: {ui_json['page']['name']}</p>
        <button id="build">Build Layout</button>
        <script>
            const uiData = {json.dumps(ui_json)};
            document.getElementById('build').onclick = () => {{
                parent.postMessage({{ pluginMessage: {{ type: 'generate', data: uiData }} }}, '*');
            }};
        </script>
        """
        with open(os.path.join(figma_dir, "ui.html"), "w") as f:
            f.write(ui_html)

        logger.info(f"Figma plugin generated at {figma_dir}/")


In [31]:
# # [CELL 9] - Main Pipeline Orchestrator
# class Exporter:
#     @staticmethod
#     def create_zip(source_dir: str, zip_path: str):
#         with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
#             for root, _, files in os.walk(source_dir):
#                 for file in files:
#                     file_path = os.path.join(root, file)
#                     arcname = os.path.relpath(file_path, source_dir)
#                     zipf.write(file_path, arcname)
#         logger.info(f"ZIP created at {zip_path}")

# class ConverterPipeline:
#     def __init__(self, config: Dict):
#         self.config = config
#         self.ocr = OCRProcessor(langs=config['ocr_lang'])

#     def process_file(self, image_path: str):
#         filename = os.path.basename(image_path)
#         logger.info(f"Processing: {filename}")

#         try:
#             # 1. Preprocess
#             prep_data = ImageProcessor.preprocess(image_path, self.config["output_path"])

#             # 2. OCR
#             ocr_data = self.ocr.extract_text(image_path) if self.config["ocr_enabled"] else []

#             # 3. UI Detection
#             elements = UIElementDetector.detect_elements(prep_data, ocr_data)

#             # 4. JSON Generation
#             ui_json = UIJSONBuilder.build(prep_data["dimensions"], elements, filename)
#             json_path = os.path.join(self.config["output_path"], f"{filename}_ui.json")
#             with open(json_path, "w") as f:
#                 json.dump(ui_json, f, indent=2)

#             # 5. Output Generation
#             if self.config["generate_html"]:
#                 HTMLGenerator.generate(image_path, self.config["output_path"])
#             if self.config["generate_react"]:
#                 ReactTailwindGenerator.generate(ui_json, self.config["output_path"])
#             if self.config["generate_figma"]:
#                 FigmaGenerator.generate_plugin(ui_json, self.config["output_path"])

#             # 6. Validation (Simulated rendering loop)
#             similarity = Validator.validate_visual(prep_data["image"], "mock_html_path")

#             return True, similarity

#         except Exception as e:
#             logger.error(f"Failed processing {filename}: {str(e)}")
#             return False, 0.0


In [32]:
# # [CELL 10] - UI Dashboard and Execution
# output_widget = widgets.Output()

# # UI Elements
# title = widgets.HTML("<h2>🎨 Screenshot to Editable UI Converter</h2>")
# source_dropdown = widgets.Dropdown(options=[('Google Drive', 'drive'), ('Upload Files', 'upload')], value='upload', description='Source:')
# drive_path_input = widgets.Text(value='/kaggle/working/ui_converter/input/', description='Drive Path:', layout=widgets.Layout(width='400px'))
# upload_button = widgets.FileUpload(accept='image/*', multiple=True, description='Upload Images')
# run_button = widgets.Button(description='🚀 Generate UI', button_style='success', layout=widgets.Layout(width='200px'))
# progress_bar = widgets.IntProgress(value=0, min=0, max=10, description='Progress:', bar_style='info', layout=widgets.Layout(width='400px'))

# # State Management
# def update_ui(*args):
#     if source_dropdown.value == 'drive':
#         drive_path_input.disabled = False
#         upload_button.disabled = True
#     else:
#         drive_path_input.disabled = True
#         upload_button.disabled = False

# source_dropdown.observe(update_ui, 'value')
# update_ui()

# # Execution Logic
# def on_run_clicked(b):
#     with output_widget:
#         clear_output()
#         CONFIG["input_source"] = source_dropdown.value
#         CONFIG["drive_path"] = drive_path_input.value

#         manager = InputManager(CONFIG)
#         files_to_process = []

#         if CONFIG["input_source"] == 'upload':
#             if not upload_button.value:
#                 print("⚠️ Please upload files first.")
#                 return
#             files_to_process = manager.handle_upload(upload_button.value)
#         else:
#             manager.mount_drive()
#             files_to_process = manager.fetch_from_drive()

#         if not files_to_process:
#             print("⚠️ No valid screenshots found to process.")
#             return

#         pipeline = ConverterPipeline(CONFIG)
#         progress_bar.max = len(files_to_process)
#         progress_bar.value = 0

#         success_count = 0
#         for img_path in files_to_process:
#             success, sim = pipeline.process_file(img_path)
#             if success:
#                 success_count += 1
#             progress_bar.value += 1

#         # Zip Creation
#         zip_file = "/kaggle/working/screenshot-to-ui.zip"
#         Exporter.create_zip(CONFIG["output_path"], zip_file)

#         print("\n" + "="*40)
#         print("🎉 SCREENSHOT → EDITABLE UI COMPLETE")
#         print("="*40)
#         print(f"Screenshots processed: {len(files_to_process)}")
#         print(f"Successful: {success_count} | Failed: {len(files_to_process) - success_count}")
#         print("HTML generated: YES")
#         print("React generated: YES")
#         print("Tailwind generated: YES")
#         print("Figma package (Plugin) generated: YES")
#         print(f"Output Directory: {CONFIG['output_path']}")

#         # Display Download Button
#         display(HTML(f'''
#             <a href="/kaggle/working/screenshot-to-ui.zip" download>
#                 <button style="padding: 10px 20px; background-color: #007bff; color: white; border: none; border-radius: 5px; cursor: pointer;">
#                     ⬇️ Download screenshot-to-ui.zip
#                 </button>
#             </a>
#         '''))

# run_button.on_click(on_run_clicked)

# # Display Dashboard
# display(title)
# display(widgets.HBox([source_dropdown]))
# display(widgets.HBox([drive_path_input, upload_button]))
# display(widgets.HTML("<hr>"))
# display(run_button)
# display(progress_bar)
# display(output_widget)


Update Code For 9 and 10


In [42]:
# ============================================================
# 🚀 FINAL DEVSTRAL FIX — KAGGLE 2×T4
#    CLEAN DISK → DOWNLOAD LOCALLY → START LLAMA.CPP
# ============================================================

import os
import sys
import time
import shutil
import subprocess
import urllib.request

# ============================================================
# CONFIG
# ============================================================

REPO = "unsloth/Devstral-Small-2-24B-Instruct-2512-GGUF"

MODEL_ID = "mistralai/Devstral-Small-2-24B-Instruct-2512"

MODEL_DIR = "/kaggle/working/devstral-model"

HOST = "127.0.0.1"
PORT = 8000

API_URL = f"http://{HOST}:{PORT}/v1/models"

LOG_FILE = "/kaggle/working/devstral-llama.log"

LLAMA_BIN = "/root/.local/bin/llama"


# ============================================================
# HEADER
# ============================================================

print("=" * 72)
print("🚀 DEVSTRAL SMALL 2 — DISK CLEANUP + 2×T4")
print("=" * 72)


# ============================================================
# 1. STOP OLD SERVERS
# ============================================================

print("\n[1/8] STOPPING OLD SERVERS")

subprocess.run(
    ["pkill", "-f", "vllm.entrypoints.openai.api_server"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
    check=False
)

subprocess.run(
    ["pkill", "-f", "llama-server"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
    check=False
)

subprocess.run(
    ["pkill", "-f", "llama serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
    check=False
)

time.sleep(3)

print("✅ Old model servers stopped.")


# ============================================================
# 2. CLEAN FAILED DEVSTRAL DOWNLOADS
# ============================================================

print("\n[2/8] CLEANING FAILED DEVSTRAL DOWNLOADS")

paths_to_remove = [
    # Previous llama.cpp HF cache
    "/kaggle/working/llama-cache",

    # Previous manually downloaded model directory
    "/kaggle/working/devstral-model",

    # Old failed model cache under root
    "/root/.cache/huggingface/hub/models--unsloth--Devstral-Small-2-24B-Instruct-2512-GGUF",
]

for path in paths_to_remove:

    if os.path.exists(path):

        try:

            size = shutil.disk_usage(path) if os.path.isdir(path) else None

            print(
                f"🗑️ Removing: {path}"
            )

            shutil.rmtree(
                path,
                ignore_errors=True
            )

        except Exception as e:

            print(
                f"⚠️ Could not remove {path}: {e}"
            )


# ============================================================
# 3. CLEAN SAFE TEMP CACHES
# ============================================================

print("\n[3/8] CLEANING SAFE TEMP CACHES")

safe_cache_dirs = [
    "/root/.cache/pip",
    "/tmp/pip-ephem-wheel-cache",
    "/tmp/pip-unpack",
    "/tmp/pip-build-tracker",
]

for path in safe_cache_dirs:

    if os.path.exists(path):

        try:

            print(
                "🧹",
                path
            )

            shutil.rmtree(
                path,
                ignore_errors=True
            )

        except Exception:
            pass


# Remove only old generated ZIP/log files.
for path in [
    "/kaggle/working/screenshot-to-ui.zip",
]:

    if os.path.isfile(path):

        try:

            os.remove(path)
            print(
                "🗑️ Removed:",
                path
            )

        except Exception:
            pass


# ============================================================
# 4. DISK CHECK
# ============================================================

print("\n[4/8] DISK STATUS")

disk = shutil.disk_usage(
    "/kaggle/working"
)

total_gb = disk.total / (1024 ** 3)
used_gb = disk.used / (1024 ** 3)
free_gb = disk.free / (1024 ** 3)

print(
    f"Total : {total_gb:.2f} GB"
)

print(
    f"Used  : {used_gb:.2f} GB"
)

print(
    f"Free  : {free_gb:.2f} GB"
)


# ============================================================
# MODEL SELECTION
# ============================================================
#
# Q4_K_M  ≈ 14.3 GB
# Q3_K_M  ≈ 11.5 GB
# mmproj-F16 ≈ 0.878 GB
#
# We need extra room for metadata/temp files.
#
# Select Q4 only when >= 17 GB free.
# Otherwise use Q3_K_M.
# ============================================================

if free_gb >= 17.0:

    QUANT = "Q4_K_M"

    MODEL_FILE = (
        "Devstral-Small-2-24B-Instruct-2512-Q4_K_M.gguf"
    )

    required_gb = 16.0

else:

    QUANT = "Q3_K_M"

    MODEL_FILE = (
        "Devstral-Small-2-24B-Instruct-2512-Q3_K_M.gguf"
    )

    required_gb = 13.0


print(
    "\n🎯 Selected quantization:",
    QUANT
)

print(
    "📦 Model file:",
    MODEL_FILE
)

print(
    f"💾 Required free space: ~{required_gb:.1f} GB"
)


if free_gb < required_gb:

    raise RuntimeError(
        f"\n❌ Still not enough disk space.\n"
        f"Free: {free_gb:.2f} GB\n"
        f"Required: ~{required_gb:.1f} GB\n\n"
        "Delete more files from /kaggle/working "
        "before downloading Devstral."
    )


# ============================================================
# 5. CREATE MODEL DIRECTORY
# ============================================================

print("\n[5/8] PREPARING MODEL DIRECTORY")

os.makedirs(
    MODEL_DIR,
    exist_ok=True
)

MODEL_PATH = os.path.join(
    MODEL_DIR,
    MODEL_FILE
)

MMPROJ_FILE = "mmproj-F16.gguf"

MMPROJ_PATH = os.path.join(
    MODEL_DIR,
    MMPROJ_FILE
)


# ============================================================
# 6. DOWNLOAD WITH HUGGING FACE HUB
# ============================================================

print("\n[6/8] DOWNLOADING MODEL")

try:

    from huggingface_hub import hf_hub_download

except ImportError:

    print(
        "⏳ Installing huggingface_hub..."
    )

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "huggingface_hub"
        ],
        check=True
    )

    from huggingface_hub import hf_hub_download


# ------------------------------------------------------------
# Download main model
# ------------------------------------------------------------

if os.path.isfile(MODEL_PATH):

    print(
        "✅ Model already exists."
    )

else:

    print(
        f"\n⏳ Downloading {QUANT}..."
    )

    print(
        "Repository:",
        REPO
    )

    print(
        "File:",
        MODEL_FILE
    )

    hf_hub_download(
        repo_id=REPO,
        filename=MODEL_FILE,
        local_dir=MODEL_DIR,
        local_dir_use_symlinks=False,
    )

    print(
        "✅ Model download complete."
    )


# ------------------------------------------------------------
# Download vision projector
# ------------------------------------------------------------

if os.path.isfile(MMPROJ_PATH):

    print(
        "✅ Vision projector already exists."
    )

else:

    print(
        "\n⏳ Downloading vision projector..."
    )

    hf_hub_download(
        repo_id=REPO,
        filename=MMPROJ_FILE,
        local_dir=MODEL_DIR,
        local_dir_use_symlinks=False,
    )

    print(
        "✅ Vision projector download complete."
    )


# ============================================================
# FILE SIZE CHECK
# ============================================================

model_size = (
    os.path.getsize(MODEL_PATH)
    / (1024 ** 3)
)

mmproj_size = (
    os.path.getsize(MMPROJ_PATH)
    / (1024 ** 3)
)

print(
    "\n📦 DOWNLOADED FILES"
)

print(
    f"Model  : {model_size:.2f} GB"
)

print(
    f"mmproj : {mmproj_size:.2f} GB"
)

print(
    f"Total  : {model_size + mmproj_size:.2f} GB"
)


# ============================================================
# 7. CHECK LLAMA
# ============================================================

print("\n[7/8] CHECKING LLAMA.CPP")

if not os.path.isfile(LLAMA_BIN):

    raise RuntimeError(
        f"❌ llama.cpp not found: {LLAMA_BIN}"
    )

version = subprocess.run(
    [
        LLAMA_BIN,
        "version"
    ],
    capture_output=True,
    text=True
)

print(
    version.stdout.strip()
)


# ============================================================
# START SERVER
# ============================================================

COMMAND = [
    LLAMA_BIN,
    "serve",

    "--model",
    MODEL_PATH,

    "--mmproj",
    MMPROJ_PATH,

    "--host",
    HOST,

    "--port",
    str(PORT),

    "--alias",
    MODEL_ID,

    "--ctx-size",
    "8192",

    "--parallel",
    "1",

    # Both T4 GPUs
    "--n-gpu-layers",
    "all",

    "--split-mode",
    "layer",

    "--tensor-split",
    "1,1",

    # Stable T4 setting
    "--flash-attn",
    "off",

    # Vision projector on GPU
    "--mmproj-offload",

    # Mistral chat template
    "--jinja",
]


print(
    "\n🔥 STARTING DEVSTRAL"
)

print(
    "Quantization:",
    QUANT
)

print(
    "GPUs: 2× Tesla T4"
)

print(
    "Tensor split: 1,1"
)

print(
    "Context: 8192"
)

print(
    "\nCOMMAND:"
)

print(
    " ".join(COMMAND)
)


# ============================================================
# LOG
# ============================================================

with open(
    LOG_FILE,
    "w",
    encoding="utf-8"
) as f:

    f.write("")


log_handle = open(
    LOG_FILE,
    "a",
    buffering=1,
    encoding="utf-8"
)


# ============================================================
# START
# ============================================================

process = subprocess.Popen(
    COMMAND,
    stdout=log_handle,
    stderr=subprocess.STDOUT,
    text=True
)


# ============================================================
# WAIT FOR API
# ============================================================

print(
    "\n⏳ Loading Devstral onto GPUs..."
)

started = time.time()

timeout = 900

ready = False

last_gpu_check = 0

while (
    time.time() - started
    < timeout
):

    # --------------------------------------------------------
    # SERVER CRASH
    # --------------------------------------------------------

    if process.poll() is not None:

        log_handle.flush()

        print(
            "\n❌ SERVER EXITED"
        )

        print(
            "\n========== LAST LOG ==========\n"
        )

        try:

            with open(
                LOG_FILE,
                "r",
                encoding="utf-8",
                errors="ignore"
            ) as f:

                print(
                    f.read()[-20000:]
                )

        except Exception as e:

            print(
                "Log read error:",
                e
            )

        raise RuntimeError(
            "Devstral failed during startup."
        )

    # --------------------------------------------------------
    # API TEST
    # --------------------------------------------------------

    try:

        with urllib.request.urlopen(
            API_URL,
            timeout=3
        ) as response:

            if response.status == 200:

                ready = True
                break

    except Exception:
        pass

    elapsed = int(
        time.time() - started
    )

    print(
        f"\r⏳ Loading... {elapsed}s",
        end="",
        flush=True
    )

    # --------------------------------------------------------
    # GPU CHECK
    # --------------------------------------------------------

    if elapsed - last_gpu_check >= 20:

        last_gpu_check = elapsed

        print(
            "\n"
        )

        gpu = subprocess.run(
            [
                "nvidia-smi",
                "--query-gpu=index,name,memory.used,memory.total,utilization.gpu",
                "--format=csv,noheader"
            ],
            capture_output=True,
            text=True
        )

        print(
            gpu.stdout.strip()
        )

    time.sleep(3)


# ============================================================
# FINAL FAILURE
# ============================================================

if not ready:

    log_handle.flush()

    print(
        "\n❌ STARTUP TIMEOUT"
    )

    try:

        with open(
            LOG_FILE,
            "r",
            encoding="utf-8",
            errors="ignore"
        ) as f:

            print(
                f.read()[-20000:]
            )

    except:
        pass

    raise RuntimeError(
        "Devstral did not become ready."
    )


# ============================================================
# SUCCESS
# ============================================================

print(
    "\n\n" + "=" * 72
)

print(
    "🎉 DEVSTRAL SMALL 2 IS READY"
)

print(
    "=" * 72
)

print(
    "Model:",
    MODEL_ID
)

print(
    "Quantization:",
    QUANT
)

print(
    "Backend: llama.cpp"
)

print(
    "Vision: ENABLED"
)

print(
    "GPU: 2× Tesla T4"
)

print(
    "API:",
    f"http://{HOST}:{PORT}/v1"
)

print(
    "Log:",
    LOG_FILE
)


# ============================================================
# FINAL GPU STATUS
# ============================================================

print(
    "\n🔥 FINAL GPU MEMORY:"
)

subprocess.run(
    [
        "nvidia-smi",
        "--query-gpu=index,name,memory.used,memory.total,utilization.gpu",
        "--format=csv"
    ],
    check=False
)


# ============================================================
# VARIABLES FOR EXISTING GENERATOR
# ============================================================

DEVSTRAL_BASE_URL = (
    f"http://{HOST}:{PORT}/v1"
)

DEVSTRAL_MODEL = MODEL_ID

try:

    CONFIG["devstral_base_url"] = (
        DEVSTRAL_BASE_URL
    )

    CONFIG["devstral_model"] = (
        DEVSTRAL_MODEL
    )

    CONFIG["ai_engine"] = (
        "devstral_24b"
    )

except Exception:
    pass

print(
    "\n✅ Existing Screenshot → UI generator can use:"
)

print(
    "   Base URL:",
    DEVSTRAL_BASE_URL
)

print(
    "   Model:",
    DEVSTRAL_MODEL
)

🚀 DEVSTRAL SMALL 2 — DISK CLEANUP + 2×T4

[1/8] STOPPING OLD SERVERS
✅ Old model servers stopped.

[2/8] CLEANING FAILED DEVSTRAL DOWNLOADS
🗑️ Removing: /kaggle/working/llama-cache

[3/8] CLEANING SAFE TEMP CACHES
🧹 /root/.cache/pip
🗑️ Removed: /kaggle/working/screenshot-to-ui.zip

[4/8] DISK STATUS
Total : 19.52 GB
Used  : 5.62 GB
Free  : 13.88 GB

🎯 Selected quantization: Q3_K_M
📦 Model file: Devstral-Small-2-24B-Instruct-2512-Q3_K_M.gguf
💾 Required free space: ~13.0 GB

[5/8] PREPARING MODEL DIRECTORY

[6/8] DOWNLOADING MODEL

⏳ Downloading Q3_K_M...
Repository: unsloth/Devstral-Small-2-24B-Instruct-2512-GGUF
File: Devstral-Small-2-24B-Instruct-2512-Q3_K_M.gguf


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:208: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
2026-09-21 13:44:25,463 | INFO | HTTP Request: HEAD https://huggingface.co/unsloth/Devstral-Small-2-24B-Instruct-2512-GGUF/resolve/main/Devstral-Small-2-24B-Instruct-2512-Q3_K_M.gguf "HTTP/1.1 302 Found"
2026-09-21 13:44:25,547 | INFO | HTTP Request: GET https://huggingface.co/api/models/unsloth/Devstral-Small-2-24B-Instruct-2512-GGUF/xet-read-token/6e458b8add42681bfd023de5eab93637694aaf82 "HTTP/1.1 200 OK"


Devstral-Small-2-24B-Instruct-2512-Q3_K_(…): reconstructing file:   0%|          |  0.00B / 11.5GB            

Devstral-Small-2-24B-Instruct-2512-Q3_K_(…): downloading bytes:           |  0.00B            

2026-09-21 13:46:59,537 | INFO | HTTP Request: HEAD https://huggingface.co/unsloth/Devstral-Small-2-24B-Instruct-2512-GGUF/resolve/main/mmproj-F16.gguf "HTTP/1.1 302 Found"
2026-09-21 13:46:59,538 | WARNING | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


✅ Model download complete.

⏳ Downloading vision projector...


mmproj-F16.gguf: reconstructing file:   0%|          |  0.00B /  878MB            

mmproj-F16.gguf: downloading bytes:           |  0.00B            

✅ Vision projector download complete.

📦 DOWNLOADED FILES
Model  : 10.69 GB
mmproj : 0.82 GB
Total  : 11.50 GB

[7/8] CHECKING LLAMA.CPP
version: 0.4.1-dev (build 11046, commit 60081bb2b)
built with Clang 19.1.7 for Linux x86_64

🔥 STARTING DEVSTRAL
Quantization: Q3_K_M
GPUs: 2× Tesla T4
Tensor split: 1,1
Context: 8192

COMMAND:
/root/.local/bin/llama serve --model /kaggle/working/devstral-model/Devstral-Small-2-24B-Instruct-2512-Q3_K_M.gguf --mmproj /kaggle/working/devstral-model/mmproj-F16.gguf --host 127.0.0.1 --port 8000 --alias mistralai/Devstral-Small-2-24B-Instruct-2512 --ctx-size 8192 --parallel 1 --n-gpu-layers all --split-mode layer --tensor-split 1,1 --flash-attn off --mmproj-offload --jinja

⏳ Loading Devstral onto GPUs...
⏳ Loading... 6s

🎉 DEVSTRAL SMALL 2 IS READY
Model: mistralai/Devstral-Small-2-24B-Instruct-2512
Quantization: Q3_K_M
Backend: llama.cpp
Vision: ENABLED
GPU: 2× Tesla T4
API: http://127.0.0.1:8000/v1
Log: /kaggle/working/devstral-llama.log

🔥 FINAL GPU ME

In [54]:
# ============================================================
# 🎨 SCREENSHOT → COMPLETE PROJECT GENERATOR
#
# LOCAL MODEL:
#   mistralai/Devstral-Small-2-24B-Instruct-2512
#
# FEATURES:
#   • User chooses ONE framework / stack
#   • User provides project instruction
#   • Screenshot analyzed by Devstral
#   • AI generates COMPLETE PROJECT
#   • Multiple files supported
#   • Framework-specific structure
#   • package.json / config / source / components / styles
#   • Assets directory
#   • README
#   • ZIP export
#   • No "ScreenshotUI.jsx only" limitation
# ============================================================

import os
import re
import json
import base64
import shutil
import time
import zipfile
import requests
import mimetypes

import ipywidgets as widgets

from IPython.display import (
    display,
    HTML,
    clear_output,
    FileLink
)


# ============================================================
# 1. CONFIG
# ============================================================

BASE_URL = globals().get(
    "BASE_URL",
    "http://127.0.0.1:8000/v1"
)

MODEL = globals().get(
    "MODEL",
    "mistralai/Devstral-Small-2-24B-Instruct-2512"
)

PROJECT_ROOT = (
    "/kaggle/working/generated_project"
)

os.makedirs(
    PROJECT_ROOT,
    exist_ok=True
)


# ============================================================
# 2. HELPERS
# ============================================================

def image_to_data_uri(path):

    if not path:
        return None

    if not os.path.exists(path):
        return None

    mime, _ = mimetypes.guess_type(
        path
    )

    if not mime:
        mime = "image/png"

    with open(
        path,
        "rb"
    ) as f:

        encoded = base64.b64encode(
            f.read()
        ).decode("ascii")

    return (
        "data:" +
        mime +
        ";base64," +
        encoded
    )


def clean_json_response(text):

    if not text:
        return ""

    text = str(
        text
    ).strip()

    # Remove fenced JSON
    match = re.search(
        r"```(?:json)?\s*([\s\S]*?)\s*```",
        text,
        re.IGNORECASE
    )

    if match:

        return match.group(
            1
        ).strip()

    # Try direct object
    start = text.find("{")
    end = text.rfind("}")

    if (
        start >= 0
        and end > start
    ):

        return text[
            start:end + 1
        ].strip()

    return text


def safe_project_path(
    relative_path
):

    relative_path = str(
        relative_path
    ).replace(
        "\\",
        "/"
    ).strip()

    relative_path = relative_path.lstrip(
        "/"
    )

    # Prevent traversal
    parts = []

    for part in relative_path.split("/"):

        if part in {
            "",
            ".",
            ".."
        }:
            continue

        parts.append(
            part
        )

    clean = "/".join(
        parts
    )

    if not clean:
        raise ValueError(
            "Invalid empty project path."
        )

    return clean


def write_project_files(
    project
):

    files = project.get(
        "files",
        []
    )

    if not isinstance(
        files,
        list
    ):
        raise ValueError(
            "AI project manifest does not contain a valid files list."
        )

    written = []

    for item in files:

        if not isinstance(
            item,
            dict
        ):
            continue

        path = item.get(
            "path"
        )

        content = item.get(
            "content"
        )

        if not path:
            continue

        if content is None:
            content = ""

        path = safe_project_path(
            path
        )

        destination = os.path.join(
            PROJECT_ROOT,
            path
        )

        # Final path safety check
        root_abs = os.path.abspath(
            PROJECT_ROOT
        )

        dest_abs = os.path.abspath(
            destination
        )

        if not (
            dest_abs == root_abs
            or dest_abs.startswith(
                root_abs + os.sep
            )
        ):

            raise ValueError(
                "Unsafe generated path: " +
                path
            )

        directory = os.path.dirname(
            destination
        )

        os.makedirs(
            directory,
            exist_ok=True
        )

        with open(
            destination,
            "w",
            encoding="utf-8"
        ) as f:

            f.write(
                str(content)
            )

        written.append(
            destination
        )

    if not written:

        raise ValueError(
            "AI did not generate any project files."
        )

    return written


def create_zip():

    zip_path = (
        "/kaggle/working/"
        "screenshot-ui-project.zip"
    )

    if os.path.exists(
        zip_path
    ):
        os.remove(
            zip_path
        )

    with zipfile.ZipFile(
        zip_path,
        "w",
        zipfile.ZIP_DEFLATED
    ) as z:

        for root, dirs, files in os.walk(
            PROJECT_ROOT
        ):

            for filename in files:

                full_path = os.path.join(
                    root,
                    filename
                )

                arcname = os.path.relpath(
                    full_path,
                    PROJECT_ROOT
                )

                z.write(
                    full_path,
                    arcname
                )

    return zip_path


def project_file_tree():

    result = []

    for root, dirs, files in os.walk(
        PROJECT_ROOT
    ):

        dirs[:] = [
            d for d in dirs
            if d != "__pycache__"
        ]

        for filename in files:

            path = os.path.join(
                root,
                filename
            )

            result.append(
                os.path.relpath(
                    path,
                    PROJECT_ROOT
                ).replace(
                    "\\",
                    "/"
                )
            )

    return sorted(
        result
    )


# ============================================================
# 3. SCREENSHOT
# ============================================================

def get_uploaded_image():

    value = screenshot_upload.value

    if not value:
        return None, None

    if isinstance(
        value,
        (tuple, list)
    ):

        if len(value) == 0:
            return None, None

        item = value[0]

        if isinstance(
            item,
            dict
        ):

            return (
                item.get(
                    "name",
                    "screenshot.png"
                ),
                item.get(
                    "content",
                    b""
                )
            )

    if isinstance(
        value,
        dict
    ):

        if not value:
            return None, None

        name, item = next(
            iter(
                value.items()
            )
        )

        if isinstance(
            item,
            dict
        ):

            return (
                item.get(
                    "name",
                    name
                ),
                item.get(
                    "content",
                    b""
                )
            )

        return (
            name,
            item
        )

    return None, None


# ============================================================
# 4. LOCAL DEVSTRAL
# ============================================================

def call_devstral_project(
    screenshot_path,
    framework,
    instruction,
    project_name
):

    image_uri = image_to_data_uri(
        screenshot_path
    )

    system_prompt = """
You are a principal frontend architect and
expert screenshot-to-production-project engineer.

You generate COMPLETE software projects.

You do NOT generate only one component.

The user's selected framework/stack is mandatory.

Return a machine-readable JSON project manifest.
"""


    user_prompt = f"""
Create a COMPLETE production-ready frontend project
from the uploaded screenshot.

============================================================
PROJECT
============================================================

Project name:
{project_name}

Framework / Stack:
{framework}

User instruction:
{instruction}

============================================================
MOST IMPORTANT REQUIREMENT
============================================================

Generate ALL files required for the selected framework/stack.

Do NOT return only:

ScreenshotUI.jsx

Do NOT return only:

App.jsx

Do NOT return only one HTML file.

Return the COMPLETE PROJECT STRUCTURE.

The generated project must be something the user can
save into a directory and continue developing.

============================================================
FRAMEWORK RULE
============================================================

The user's framework/stack is authoritative:

{framework}

Use exactly that framework/stack.

Do not silently replace it.

Do not convert Next.js to React.

Do not convert React to Vue.

Do not convert Vue to Svelte.

Do not add Tailwind unless requested.

Do not add shadcn/ui unless requested.

Do not add Bootstrap unless requested.

Do not invent a different architecture.

============================================================
PROJECT FILES
============================================================

Generate every file that is actually needed.

Depending on the selected framework this may include:

- package.json
- lockfile only when useful
- tsconfig.json
- framework configuration
- build configuration
- Tailwind configuration
- PostCSS configuration
- shadcn configuration
- source entry files
- app/page files
- layouts
- global CSS
- reusable components
- UI components
- utility files
- hooks
- data files
- asset files where text-based assets are possible
- public files
- README.md

Do not create unnecessary files.

Do not create fake files just to make the tree larger.

============================================================
SCREENSHOT RECONSTRUCTION
============================================================

Analyze the screenshot carefully.

Reconstruct:

- complete page structure
- header
- navigation
- sidebar
- footer
- hero
- sections
- cards
- grids
- forms
- buttons
- typography
- spacing
- padding
- margins
- gaps
- colors
- borders
- radius
- shadows
- icons
- images
- alignment
- responsive behavior
- visual hierarchy

Preserve visible text.

Do not invent content that is not visually justified.

============================================================
USER INSTRUCTION
============================================================

Apply this instruction to the project:

{instruction}

The instruction may change:

- layout
- framework configuration
- component architecture
- styling
- responsiveness
- interactions
- themes
- content
- components
- dependencies

Apply it consistently across the entire project.

============================================================
CODE QUALITY
============================================================

Build a coherent project.

Files must reference each other correctly.

Imports must use correct paths.

Components must exist if imported.

Utility functions must exist if imported.

Stylesheets must exist if imported.

Configuration must match the framework.

package.json must contain the dependencies required
by the generated source.

Avoid unnecessary dependencies.

Do not use broken placeholder imports.

Do not use nonexistent local files.

Do not use lorem ipsum.

Do not leave TODO placeholders.

Do not omit code.

============================================================
ASSETS
============================================================

For screenshot images/icons:

- Prefer reusable SVG when an icon can be represented
  as SVG.
- For decorative image placeholders, use appropriate
  generated local SVG assets when possible.
- Do not use random external URLs unless necessary.
- Keep assets inside public/assets when possible.

============================================================
RESPONSIVENESS
============================================================

Implement:

- desktop
- tablet
- mobile

using the selected framework's normal responsive system.

============================================================
OUTPUT FORMAT
============================================================

Return EXACTLY this JSON structure:

{{
  "project_name": "string",
  "framework": "string",
  "description": "string",
  "files": [
    {{
      "path": "package.json",
      "content": "FULL FILE CONTENT"
    }},
    {{
      "path": "src/App.tsx",
      "content": "FULL FILE CONTENT"
    }}
  ]
}}

IMPORTANT:

- Every file must contain its COMPLETE content.
- Never use "...".
- Never say "same as above".
- Never omit unchanged code.
- Never include Markdown outside the JSON.
- Never wrap JSON in Markdown fences.
- Return valid JSON only.
"""


    content = [
        {
            "type": "text",
            "text": user_prompt
        }
    ]


    if image_uri:

        content.append(
            {
                "type": "image_url",
                "image_url": {
                    "url": image_uri
                }
            }
        )


    payload = {

        "model": MODEL,

        "messages": [

            {
                "role":
                    "system",

                "content":
                    system_prompt
            },

            {
                "role":
                    "user",

                "content":
                    content
            }
        ],

        "temperature":
            0.08,

        "top_p":
            0.90,

        "max_tokens":
            30000,

        "stream":
            False
    }


    response = requests.post(
        BASE_URL +
        "/chat/completions",

        json=payload,

        timeout=1800
    )


    response.raise_for_status()


    data = response.json()


    if "choices" not in data:

        raise RuntimeError(
            "Invalid Devstral response:\n"
            + json.dumps(
                data,
                indent=2
            )[:15000]
        )


    content = (
        data["choices"][0]
        ["message"]
        ["content"]
    )


    if isinstance(
        content,
        list
    ):

        parts = []

        for item in content:

            if isinstance(
                item,
                dict
            ):

                if item.get(
                    "type"
                ) == "text":

                    parts.append(
                        item.get(
                            "text",
                            ""
                        )
                    )

            elif isinstance(
                item,
                str
            ):

                parts.append(
                    item
                )

        content = "\n".join(
            parts
        )


    raw_json = clean_json_response(
        content
    )


    try:

        project = json.loads(
            raw_json
        )

    except Exception as exc:

        raise RuntimeError(
            "Devstral did not return valid JSON.\n\n"
            + str(exc)
            + "\n\n"
            + raw_json[:20000]
        )


    if not isinstance(
        project,
        dict
    ):

        raise RuntimeError(
            "Project manifest must be a JSON object."
        )


    if not isinstance(
        project.get(
            "files"
        ),
        list
    ):

        raise RuntimeError(
            "Project manifest does not contain "
            "a valid files array."
        )


    return project


# ============================================================
# 5. WIDGETS
# ============================================================

header = widgets.HTML(
    """
    <div style="
        margin:20px 0 15px;
        padding:20px;
        border-radius:12px;
        background:linear-gradient(
            135deg,
            #111827,
            #1d4ed8
        );
        color:white;
    ">

        <h2 style="
            margin:0 0 7px;
        ">
            🎨 Screenshot → Complete Project
        </h2>

        <div style="
            font-size:13px;
            opacity:.92;
        ">
            Local Devstral Small 2 · Multi-file
            framework-aware project generation
        </div>

    </div>
    """
)


screenshot_upload = widgets.FileUpload(
    accept=(
        "image/png,"
        "image/jpeg,"
        "image/webp"
    ),

    multiple=False,

    description="📷 Upload Screenshot",

    layout=widgets.Layout(
        width="260px"
    )
)


project_name_input = widgets.Text(
    value="screenshot-ui",

    description="Project Name:",

    placeholder="my-ui-project",

    layout=widgets.Layout(
        width="650px"
    ),

    style={
        "description_width":
            "initial"
    }
)


framework_input = widgets.Text(
    value=(
        "Next.js + TypeScript + "
        "Tailwind CSS + shadcn/ui"
    ),

    description="Framework / Stack:",

    placeholder=(
        "React + TypeScript + Tailwind CSS"
    ),

    layout=widgets.Layout(
        width="850px"
    ),

    style={
        "description_width":
            "initial"
    }
)


framework_info = widgets.HTML(
    """
    <div style="
        padding:11px 14px;
        margin:3px 0 14px;
        background:#eff6ff;
        border-left:4px solid #2563eb;
        border-radius:7px;
        font:13px system-ui;
    ">

        <b>ONE framework / stack.</b>

        The AI will create the complete project
        structure required by that stack.

        <br><br>

        Examples:

        <code>
        Next.js + TypeScript + Tailwind CSS + shadcn/ui
        </code>

        <br>

        <code>
        React + TypeScript + Vite + Tailwind CSS
        </code>

        <br>

        <code>
        Vue 3 + Vite + TypeScript + Tailwind CSS
        </code>

        <br>

        <code>
        SvelteKit + TypeScript
        </code>

        <br>

        <code>
        HTML + CSS + JavaScript
        </code>

    </div>
    """
)


instruction_input = widgets.Textarea(
    value="",

    description="Instruction:",

    placeholder=(
        "Describe the project changes you want.\n\n"
        "Examples:\n"
        "Create a responsive dashboard.\n"
        "Use dark mode.\n"
        "Make the sidebar collapsible.\n"
        "Add animations.\n"
        "Use shadcn/ui components.\n"
        "Keep the screenshot layout unchanged."
    ),

    layout=widgets.Layout(
        width="850px",
        height="150px"
    ),

    style={
        "description_width":
            "initial"
    }
)


generate_button = widgets.Button(
    description="🚀 Generate Complete Project",

    button_style="success",

    layout=widgets.Layout(
        width="260px",
        height="46px"
    )
)


clear_button = widgets.Button(
    description="🗑️ Clear",

    button_style="warning",

    layout=widgets.Layout(
        width="120px",
        height="46px"
    )
)


output = widgets.Output()


# ============================================================
# 6. GENERATE
# ============================================================

_running = False


def generate_project(
    _
):

    global _running

    if _running:
        return

    framework = (
        framework_input.value.strip()
    )

    instruction = (
        instruction_input.value.strip()
    )

    project_name = (
        project_name_input.value.strip()
        or "screenshot-ui"
    )


    if not framework:

        with output:

            clear_output()

            print(
                "⚠️ Please enter the framework / stack."
            )

        return


    filename, image_content = (
        get_uploaded_image()
    )


    if not filename or not image_content:

        with output:

            clear_output()

            print(
                "⚠️ Please upload a screenshot first."
            )

        return


    # Clean old generated project
    if os.path.exists(
        PROJECT_ROOT
    ):

        shutil.rmtree(
            PROJECT_ROOT
        )

    os.makedirs(
        PROJECT_ROOT,
        exist_ok=True
    )


    input_path = os.path.join(
        "/kaggle/working",
        "_ui_reference_" +
        os.path.basename(
            filename
        )
    )


    try:

        if isinstance(
            image_content,
            memoryview
        ):
            image_content = (
                image_content.tobytes()
            )

        elif isinstance(
            image_content,
            bytearray
        ):
            image_content = bytes(
                image_content
            )

        with open(
            input_path,
            "wb"
        ) as f:

            f.write(
                image_content
            )

    except Exception as exc:

        with output:

            clear_output()

            print(
                "❌ Could not save screenshot:"
            )

            print(
                repr(exc)
            )

        return


    _running = True

    generate_button.disabled = True

    old_label = (
        generate_button.description
    )

    generate_button.description = (
        "⏳ Devstral generating..."
    )


    try:

        with output:

            clear_output()

            print(
                "=" * 76
            )

            print(
                "🎨 SCREENSHOT → COMPLETE PROJECT"
            )

            print(
                "=" * 76
            )

            print(
                "\n📷 Screenshot:",
                filename
            )

            print(
                "🧩 Framework:",
                framework
            )

            print(
                "📦 Project:",
                project_name
            )

            print(
                "🧠 Model:",
                MODEL
            )

            print(
                "🔗 API:",
                BASE_URL
            )


            if instruction:

                print(
                    "\n✏️ Instruction:"
                )

                print(
                    instruction
                )


            print(
                "\n🔌 Checking local Devstral..."
            )


            health = requests.get(
                BASE_URL +
                "/models",

                timeout=8
            )


            health.raise_for_status()


            print(
                "✅ Local Devstral is reachable."
            )


            print(
                "\n🤖 Generating COMPLETE "
                "multi-file project..."
            )


            started = time.time()


            project = call_devstral_project(
                screenshot_path=input_path,
                framework=framework,
                instruction=instruction,
                project_name=project_name
            )


            elapsed = (
                time.time() -
                started
            )


            print(
                f"⏱️ AI generation: "
                f"{elapsed:.1f}s"
            )


            # ------------------------------------------------
            # Write all files
            # ------------------------------------------------

            written = write_project_files(
                project
            )


            # ------------------------------------------------
            # Add reference screenshot
            # ------------------------------------------------

            assets_dir = os.path.join(
                PROJECT_ROOT,
                "public",
                "assets"
            )

            os.makedirs(
                assets_dir,
                exist_ok=True
            )


            reference_destination = os.path.join(
                assets_dir,
                "reference-screenshot" +
                os.path.splitext(
                    filename
                )[1].lower()
            )


            shutil.copy2(
                input_path,
                reference_destination
            )


            written.append(
                reference_destination
            )


            # ------------------------------------------------
            # README metadata
            # ------------------------------------------------

            metadata_path = os.path.join(
                PROJECT_ROOT,
                "GENERATION_INFO.md"
            )


            metadata = (
                "# Screenshot UI Generation\n\n"
                "## Framework / Stack\n\n"
                + framework +
                "\n\n"
                "## Project\n\n"
                + project_name +
                "\n\n"
                "## User Instruction\n\n"
                + (
                    instruction
                    if instruction
                    else
                    "Recreate the screenshot accurately."
                )
                + "\n\n"
                "## Model\n\n"
                + MODEL +
                "\n\n"
                "## Generated Files\n\n"
            )


            for path in project_file_tree():

                metadata += (
                    "- `" +
                    path +
                    "`\n"
                )


            with open(
                metadata_path,
                "w",
                encoding="utf-8"
            ) as f:

                f.write(
                    metadata
                )


            # ------------------------------------------------
            # ZIP
            # ------------------------------------------------

            zip_path = create_zip()


            print(
                "\n✅ COMPLETE PROJECT GENERATED"
            )

            print(
                "-" * 76
            )


            project_files = project_file_tree()


            print(
                f"\n📁 Total files: "
                f"{len(project_files)}"
            )


            print(
                "\n📂 PROJECT STRUCTURE:"
            )


            for path in project_files:

                print(
                    "   " +
                    path
                )


            print(
                "\n📦 ZIP:"
            )

            print(
                zip_path
            )


            display(
                FileLink(
                    zip_path,
                    result_html_prefix=(
                        "⬇️ Download Complete Project: "
                    )
                )
            )


            print(
                "\n✅ The generated project now contains "
                "all AI-created files, not only one component."
            )


            # ------------------------------------------------
            # Show metadata
            # ------------------------------------------------

            description = project.get(
                "description",
                ""
            )

            if description:

                print(
                    "\n📝 PROJECT DESCRIPTION:"
                )

                print(
                    description
                )


    except requests.HTTPError as exc:

        with output:

            print(
                "\n❌ Devstral HTTP error:"
            )

            if exc.response is not None:

                try:

                    print(
                        exc.response.text[:15000]
                    )

                except Exception:

                    print(
                        repr(exc)
                    )

            else:

                print(
                    repr(exc)
                )


    except Exception as exc:

        with output:

            print(
                "\n❌ Project generation failed:"
            )

            print(
                repr(exc)
            )


    finally:

        _running = False

        generate_button.disabled = False

        generate_button.description = (
            old_label
        )

        try:

            os.remove(
                input_path
            )

        except Exception:
            pass


# ============================================================
# 7. CLEAR
# ============================================================

def clear_all(
    _
):

    with output:

        clear_output()

    try:

        screenshot_upload.value = ()

    except Exception:
        pass

    project_name_input.value = (
        "screenshot-ui"
    )

    instruction_input.value = ""


# ============================================================
# 8. EVENTS
# ============================================================

generate_button.on_click(
    generate_project
)

clear_button.on_click(
    clear_all
)


# ============================================================
# 9. DISPLAY
# ============================================================

display(
    header
)

display(
    screenshot_upload
)

display(
    project_name_input
)

display(
    framework_input
)

display(
    framework_info
)

display(
    instruction_input
)

display(
    widgets.HBox(
        [
            generate_button,
            clear_button
        ],
        layout=widgets.Layout(
            gap="10px"
        )
    )
)

display(
    output
)


print(
    "✅ Complete Project Generator ready."
)

print(
    "🧠 Model:",
    MODEL
)

print(
    "🔗 API:",
    BASE_URL
)

print(
    "📁 Output:",
    PROJECT_ROOT
)

print(
    "\nThe AI will generate a COMPLETE multi-file project "
    "according to the selected framework and instruction."
)

HTML(value='\n    <div style="\n        margin:20px 0 15px;\n        padding:20px;\n        border-radius:12px…

FileUpload(value=(), accept='image/png,image/jpeg,image/webp', description='📷 Upload Screenshot', layout=Layou…

Text(value='screenshot-ui', description='Project Name:', layout=Layout(width='650px'), placeholder='my-ui-proj…

Text(value='Next.js + TypeScript + Tailwind CSS + shadcn/ui', description='Framework / Stack:', layout=Layout(…

HTML(value='\n    <div style="\n        padding:11px 14px;\n        margin:3px 0 14px;\n        background:#ef…

Textarea(value='', description='Instruction:', layout=Layout(height='150px', width='850px'), placeholder='Desc…

Output()

✅ Complete Project Generator ready.
🧠 Model: mistralai/Devstral-Small-2-24B-Instruct-2512
🔗 API: http://127.0.0.1:8000/v1
📁 Output: /kaggle/working/generated_project

The AI will generate a COMPLETE multi-file project according to the selected framework and instruction.


In [58]:

# ============================================================
# CELL 11 - COMPLETE PROJECT AI REFINER + WORKING LIVE PREVIEW
#
# KAGGLE + LOCAL DEVSTRAL SMALL 2 + llama.cpp
#
# IMPORTANT:
# This cell works with the COMPLETE MULTI-FILE PROJECT produced
# by the Screenshot -> Complete Project Generator.
#
# Live Preview:
#   - React / Next.js / TSX / JSX:
#       browser-side CommonJS module runtime
#       + Babel TS/JSX compiler
#       + Tailwind CDN
#       + shadcn / lucide / next compatibility shims
#   - HTML:
#       direct HTML preview
#
# Inspect:
#   Click rendered element -> selector/details -> AI edit.
#
# AI editing:
#   Devstral receives project tree + relevant source files and
#   returns CREATE / UPDATE / DELETE operations.
# ============================================================

import os
import re
import json
import html
import shutil
import tempfile
import time
import uuid
import base64
import mimetypes
import zipfile
import requests
import ipywidgets as widgets

from IPython.display import display, HTML, clear_output, FileLink


# ============================================================
# 1. LOCAL MODEL
# ============================================================

LOCAL_API = globals().get(
    "BASE_URL",
    "http://127.0.0.1:8000/v1"
)

LOCAL_MODEL = globals().get(
    "MODEL",
    "mistralai/Devstral-Small-2-24B-Instruct-2512"
)


# ============================================================
# 2. PROJECT ROOT
# ============================================================

PROJECT_ROOT = "/kaggle/working/generated_project"

if not os.path.isdir(PROJECT_ROOT):
    fallback = "/kaggle/working/generated_ui"
    if os.path.isdir(fallback):
        PROJECT_ROOT = fallback

os.makedirs(PROJECT_ROOT, exist_ok=True)


# ============================================================
# 3. FILE HELPERS
# ============================================================

IGNORED_DIRS = {
    ".git",
    "node_modules",
    "__pycache__",
    ".next",
    "dist",
    "build",
    ".cache",
    ".preview",
}


def project_files():
    result = []

    if not os.path.isdir(PROJECT_ROOT):
        return result

    for root, dirs, files in os.walk(PROJECT_ROOT):
        dirs[:] = [
            d for d in dirs
            if d not in IGNORED_DIRS
        ]

        for name in files:
            path = os.path.join(root, name)

            if any(x in path.lower() for x in [
                ".backup_",
                ".bak",
                ".tmp",
            ]):
                continue

            result.append(
                os.path.relpath(
                    path,
                    PROJECT_ROOT
                ).replace("\\", "/")
            )

    return sorted(result)


def read_text(path):
    with open(path, "r", encoding="utf-8") as f:
        return f.read()


def absolute_path(relative):
    relative = str(relative).replace("\\", "/").lstrip("/")

    parts = [
        p for p in relative.split("/")
        if p not in ("", ".", "..")
    ]

    clean = "/".join(parts)

    root = os.path.abspath(PROJECT_ROOT)
    target = os.path.abspath(
        os.path.join(root, clean)
    )

    if not target.startswith(root + os.sep):
        raise ValueError(
            "Unsafe path: " + relative
        )

    return target


def backup(path):
    if not os.path.exists(path):
        return None

    stamp = time.strftime("%Y%m%d_%H%M%S")
    target = path + ".backup_" + stamp
    shutil.copy2(path, target)
    return target


def atomic_write(path, content):
    os.makedirs(
        os.path.dirname(path),
        exist_ok=True
    )

    fd, tmp = tempfile.mkstemp(
        prefix=".refine_",
        suffix=".tmp",
        dir=os.path.dirname(path)
    )

    try:
        with os.fdopen(
            fd,
            "w",
            encoding="utf-8"
        ) as f:
            f.write(content)
            f.flush()
            os.fsync(f.fileno())

        os.replace(tmp, path)

    except Exception:
        try:
            os.remove(tmp)
        except Exception:
            pass
        raise


# ============================================================
# 4. FRAMEWORK
# ============================================================

def previous_framework():
    for name in [
        "framework_input",
        "refine_framework_input",
        "FRAMEWORK",
        "PROJECT_FRAMEWORK",
        "SELECTED_FRAMEWORK",
    ]:
        if name not in globals():
            continue

        obj = globals()[name]

        try:
            value = obj.value if hasattr(obj, "value") else obj

            if isinstance(value, str) and value.strip():
                return value.strip()

        except Exception:
            pass

    return ""


def current_framework():
    value = refine_framework_input.value.strip()
    return value or previous_framework()


def framework_kind(framework):
    fw = framework.lower()

    if (
        "next.js" in fw
        or "nextjs" in fw
        or "react" in fw
    ):
        return "react"

    if "vue" in fw or "nuxt" in fw:
        return "vue"

    if "svelte" in fw or "sveltekit" in fw:
        return "svelte"

    if "astro" in fw:
        return "astro"

    if "angular" in fw:
        return "angular"

    if (
        "html" in fw
        or "vanilla" in fw
        or "javascript" in fw
    ):
        return "html"

    return "generic"


# ============================================================
# 5. ENTRY FILE
# ============================================================

ENTRY_CANDIDATES = [
    "app/page.tsx",
    "app/page.jsx",
    "app/page.ts",
    "app/page.js",

    "src/app/page.tsx",
    "src/app/page.jsx",

    "pages/index.tsx",
    "pages/index.jsx",

    "src/App.tsx",
    "src/App.jsx",

    "App.tsx",
    "App.jsx",

    "index.html",
]


def find_entry():
    files = project_files()

    for item in ENTRY_CANDIDATES:
        if item in files:
            return item

    scored = []

    for path in files:
        low = path.lower()
        name = os.path.basename(low)

        score = 0

        if name in {
            "page.tsx",
            "page.jsx",
            "app.tsx",
            "app.jsx",
            "index.html",
        }:
            score += 100

        if "/app/" in "/" + low:
            score += 30

        if low.endswith(
            (".tsx", ".jsx", ".html")
        ):
            score += 20

        if "components/" in low:
            score -= 20

        scored.append((score, path))

    if not scored:
        return None

    scored.sort(reverse=True)
    return scored[0][1]


# ============================================================
# 6. REFERENCE SCREENSHOT
# ============================================================

def find_reference_screenshot():
    found = []

    for root, dirs, files in os.walk(PROJECT_ROOT):
        dirs[:] = [
            d for d in dirs
            if d not in IGNORED_DIRS
        ]

        for name in files:
            ext = os.path.splitext(name)[1].lower()

            if ext not in {
                ".png",
                ".jpg",
                ".jpeg",
                ".webp",
            }:
                continue

            path = os.path.join(root, name)

            try:
                modified = os.path.getmtime(path)
            except Exception:
                modified = 0

            found.append(
                (modified, path)
            )

    if not found:
        return None

    found.sort(reverse=True)
    return found[0][1]


def image_data_uri(path):
    if not path or not os.path.exists(path):
        return None

    mime, _ = mimetypes.guess_type(path)

    if not mime:
        mime = "image/png"

    with open(path, "rb") as f:
        encoded = base64.b64encode(
            f.read()
        ).decode("ascii")

    return (
        "data:" +
        mime +
        ";base64," +
        encoded
    )


# ============================================================
# 7. PROJECT CONTEXT
# ============================================================

def import_modules(source):
    result = []

    patterns = [
        r'import\s+(?:[\s\S]*?\s+from\s+)?["\']([^"\']+)["\']',
        r'export\s+[\s\S]*?\s+from\s+["\']([^"\']+)["\']',
    ]

    for pattern in patterns:
        for match in re.finditer(
            pattern,
            source
        ):
            value = match.group(1).strip()

            if value:
                result.append(value)

    return list(dict.fromkeys(result))


def resolve_local_import(
    current_file,
    module
):
    if not module:
        return None

    if (
        module.startswith("http://")
        or module.startswith("https://")
    ):
        return None

    if module.startswith("@/"):
        base = module[2:]

    elif module.startswith("~/"):
        base = module[2:]

    elif module.startswith("src/"):
        base = module

    elif module.startswith("./") or module.startswith("../"):
        current_dir = os.path.dirname(current_file)
        base = os.path.normpath(
            os.path.join(
                current_dir,
                module
            )
        ).replace("\\", "/")

    else:
        return None

    candidates = [
        base,
        base + ".tsx",
        base + ".ts",
        base + ".jsx",
        base + ".js",
        base + ".json",
        base + ".css",
        base + ".scss",
        base + ".module.css",
        base + ".module.scss",
        base + "/index.tsx",
        base + "/index.ts",
        base + "/index.jsx",
        base + "/index.js",
    ]

    for candidate in candidates:
        try:
            absolute = absolute_path(candidate)

            if os.path.isfile(absolute):
                return candidate
        except Exception:
            pass

    return None


def relevant_context(
    entry,
    max_files=30,
    max_chars=90000
):
    if not entry:
        return []

    ordered = []
    visited = set()
    total = 0

    def visit(path):
        nonlocal total

        if path in visited:
            return

        if len(ordered) >= max_files:
            return

        if total >= max_chars:
            return

        visited.add(path)

        try:
            source = read_text(
                absolute_path(path)
            )
        except Exception:
            return

        remaining = max_chars - total

        if len(source) > remaining:
            content = source[:remaining]
        else:
            content = source

        ordered.append(
            (path, content)
        )

        total += len(content)

        for module in import_modules(source):
            local = resolve_local_import(
                path,
                module
            )

            if local:
                visit(local)

    visit(entry)

    # Always include important project files.
    for special in [
        "package.json",
        "components.json",
        "tsconfig.json",
        "next.config.ts",
        "next.config.mjs",
        "tailwind.config.ts",
        "tailwind.config.js",
        "postcss.config.mjs",
        "postcss.config.js",
        "app/globals.css",
        "src/app/globals.css",
    ]:
        if len(ordered) >= max_files:
            break

        if any(x[0] == special for x in ordered):
            continue

        absolute = os.path.join(
            PROJECT_ROOT,
            special
        )

        if not os.path.isfile(absolute):
            continue

        try:
            source = read_text(absolute)
        except Exception:
            continue

        remaining = max_chars - total

        if remaining <= 0:
            break

        content = source[:remaining]

        ordered.append(
            (special, content)
        )

        total += len(content)

    return ordered


def context_text(entry):
    blocks = []

    for path, content in relevant_context(entry):
        blocks.append(
            "\n".join([
                "==============================",
                "FILE: " + path,
                "==============================",
                content
            ])
        )

    return "\n\n".join(blocks) or "(No project context.)"


# ============================================================
# 8. INSPECTOR
# ============================================================

def inspector_script(channel):

    code = r"""
<style>

#AI_INSPECTOR_BADGE {
    position:fixed;
    top:10px;
    right:10px;
    z-index:2147483647;
    padding:8px 12px;
    background:#111827;
    color:#fff;
    border-radius:8px;
    font:600 12px system-ui,sans-serif;
    pointer-events:none;
    box-shadow:0 4px 15px rgba(0,0,0,.25);
}

.AI_INSPECTOR_HOVER {
    outline:2px dashed #2563eb !important;
    outline-offset:2px !important;
    cursor:crosshair !important;
}

.AI_INSPECTOR_SELECTED {
    outline:3px solid #ef4444 !important;
    outline-offset:3px !important;
}

</style>

<div id="AI_INSPECTOR_BADGE">
    INSPECT MODE — click an element
</div>

<script>
(function(){

    var CHANNEL = __CHANNEL__;

    var selected = null;
    var hovered = null;


    function textOf(el){

        return (
            el.innerText ||
            el.textContent ||
            ""
        )
        .replace(/\s+/g," ")
        .trim()
        .slice(0,300);
    }


    function selectorOf(el){

        if(!el){
            return "";
        }


        if(el.id){
            return "#" + el.id;
        }


        var parts = [];
        var current = el;


        while(
            current &&
            current.nodeType === 1 &&
            current !== document.documentElement
        ){

            var part =
                current.tagName.toLowerCase();


            if(
                current.classList &&
                current.classList.length
            ){

                var classes =
                    Array.from(
                        current.classList
                    )
                    .filter(
                        function(c){
                            return !c.includes(
                                "AI_INSPECTOR"
                            );
                        }
                    )
                    .slice(0,2);


                if(classes.length){
                    part += "." +
                        classes.join(".");
                }
            }


            var parent =
                current.parentElement;


            if(parent){

                var siblings =
                    Array.from(
                        parent.children
                    )
                    .filter(
                        function(node){
                            return (
                                node.tagName ===
                                current.tagName
                            );
                        }
                    );


                if(siblings.length > 1){

                    part +=
                        ":nth-of-type(" +
                        (
                            siblings.indexOf(
                                current
                            ) + 1
                        ) +
                        ")";
                }
            }


            parts.unshift(part);

            current =
                parent;


            if(current === document.body){

                parts.unshift("body");
                break;
            }
        }


        return parts.join(" > ");
    }


    function detailsOf(
        el,
        selector
    ){

        var attrs = [];


        Array.from(
            el.attributes || []
        )
        .forEach(
            function(attr){

                if(
                    attr.name !== "class" &&
                    attr.name !== "style"
                ){

                    attrs.push(
                        attr.name +
                        '="' +
                        String(
                            attr.value
                        ).slice(
                            0,
                            180
                        ) +
                        '"'
                    );
                }
            }
        );


        var rect =
            el.getBoundingClientRect();


        var style =
            getComputedStyle(el);


        return [

            "TAG: " +
                el.tagName.toLowerCase(),

            "ID: " +
                (el.id || "(none)"),

            "CLASS: " +
                (
                    typeof el.className ===
                    "string"
                        ? (
                            el.className ||
                            "(none)"
                        )
                        : "(none)"
                ),

            "TEXT: " +
                (
                    textOf(el) ||
                    "(none)"
                ),

            "ATTRIBUTES: " +
                (
                    attrs.join(" | ") ||
                    "(none)"
                ),

            "RECT: " +
                Math.round(rect.x) +
                "," +
                Math.round(rect.y) +
                " " +
                Math.round(rect.width) +
                "x" +
                Math.round(rect.height),

            "DISPLAY: " +
                style.display,

            "POSITION: " +
                style.position,

            "COLOR: " +
                style.color,

            "BACKGROUND: " +
                style.backgroundColor,

            "FONT SIZE: " +
                style.fontSize,

            "SELECTOR: " +
                selector

        ].join("\\n");
    }


    document.addEventListener(
        "mousemove",
        function(event){

            var el =
                event.target;

            if(!el){
                return;
            }


            if(
                el.id ===
                "AI_INSPECTOR_BADGE"
            ){
                return;
            }


            if(
                hovered &&
                hovered !== selected
            ){

                hovered.classList.remove(
                    "AI_INSPECTOR_HOVER"
                );
            }


            hovered = el;


            if(
                hovered !== selected
            ){

                hovered.classList.add(
                    "AI_INSPECTOR_HOVER"
                );
            }

        },
        true
    );


    document.addEventListener(
        "click",
        function(event){

            var el =
                event.target;

            if(!el){
                return;
            }


            if(
                el.id ===
                "AI_INSPECTOR_BADGE"
            ){
                return;
            }


            event.preventDefault();
            event.stopPropagation();
            event.stopImmediatePropagation();


            if(selected){

                selected.classList.remove(
                    "AI_INSPECTOR_SELECTED"
                );
            }


            if(hovered){

                hovered.classList.remove(
                    "AI_INSPECTOR_HOVER"
                );
            }


            selected = el;


            selected.classList.add(
                "AI_INSPECTOR_SELECTED"
            );


            var selector =
                selectorOf(el);


            var details =
                detailsOf(
                    el,
                    selector
                );


            window.parent.postMessage(
                {
                    type:
                        "AI_INSPECTOR_SELECTION",

                    channel:
                        CHANNEL,

                    selector:
                        selector,

                    details:
                        details
                },
                "*"
            );

        },
        true
    );

})();
</script>
"""

    return code.replace(
        "__CHANNEL__",
        json.dumps(channel)
    )


# ============================================================
# 9. HTML PREVIEW
# ============================================================

def build_html_preview(
    source,
    inspect,
    channel
):

    if not inspect:
        return source

    script = inspector_script(
        channel
    )

    lower = source.lower()

    match = re.search(
        r"</body\s*>",
        lower
    )

    if match:

        return (
            source[:match.start()]
            + script
            + source[match.start():]
        )

    return source + script


# ============================================================
# 10. REACT BROWSER MODULE RUNTIME
# ============================================================

def build_react_module_map():

    extensions = {
        ".tsx",
        ".jsx",
        ".ts",
        ".js",
        ".json",
        ".css",
        ".scss",
        ".module.css",
        ".module.scss",
    }

    modules = {}

    for root, dirs, files in os.walk(
        PROJECT_ROOT
    ):

        dirs[:] = [
            d for d in dirs
            if d not in IGNORED_DIRS
        ]

        for name in files:

            if any(
                x in name
                for x in [
                    ".backup_",
                    ".bak",
                    ".tmp",
                ]
            ):
                continue

            path = os.path.join(
                root,
                name
            )

            lower = name.lower()

            if not any(
                lower.endswith(ext)
                for ext in extensions
            ):
                continue

            relative = os.path.relpath(
                path,
                PROJECT_ROOT
            ).replace("\\", "/")

            try:
                content = read_text(path)
            except Exception:
                continue

            # Avoid huge lock/config files
            if len(content) > 600000:
                continue

            modules[relative] = content

    return modules


def build_asset_map():

    assets = {}

    public_dirs = [
        os.path.join(
            PROJECT_ROOT,
            "public"
        ),
        os.path.join(
            PROJECT_ROOT,
            "public/assets"
        ),
    ]

    roots = []

    for directory in public_dirs:
        if os.path.isdir(directory):
            roots.append(directory)

    # Also allow common local asset directories.
    for candidate in [
        "assets",
        "src/assets",
        "public/images",
    ]:
        path = os.path.join(
            PROJECT_ROOT,
            candidate
        )

        if os.path.isdir(path):
            roots.append(path)

    seen = set()

    for root in roots:

        for current, dirs, files in os.walk(
            root
        ):

            for name in files:

                path = os.path.join(
                    current,
                    name
                )

                if path in seen:
                    continue

                seen.add(path)

                try:
                    size = os.path.getsize(
                        path
                    )
                except Exception:
                    continue

                # Avoid an enormous iframe document.
                if size > 4_000_000:
                    continue

                ext = os.path.splitext(
                    name
                )[1].lower()

                if ext not in {
                    ".png",
                    ".jpg",
                    ".jpeg",
                    ".webp",
                    ".gif",
                    ".svg",
                    ".ico",
                    ".avif",
                }:
                    continue

                mime, _ = mimetypes.guess_type(
                    path
                )

                if not mime:
                    mime = "application/octet-stream"

                try:

                    with open(
                        path,
                        "rb"
                    ) as f:

                        encoded = base64.b64encode(
                            f.read()
                        ).decode(
                            "ascii"
                        )

                    relative_public = os.path.relpath(
                        path,
                        PROJECT_ROOT
                    ).replace(
                        "\\",
                        "/"
                    )

                    assets[relative_public] = (
                        "data:" +
                        mime +
                        ";base64," +
                        encoded
                    )

                except Exception:
                    pass

    return assets


def make_react_preview(
    entry,
    inspect,
    channel
):

    modules = (
        build_react_module_map()
    )

    assets = (
        build_asset_map()
    )

    # Keep generated page and full project source available.
    if entry not in modules:

        raise RuntimeError(
            "Entry file is not in the preview module map: " +
            entry
        )

    module_json = json.dumps(
        modules,
        ensure_ascii=False
    )

    asset_json = json.dumps(
        assets,
        ensure_ascii=False
    )

    global_css = ""

    for relative in [
        "app/globals.css",
        "src/app/globals.css",
        "src/index.css",
        "src/App.css",
        "globals.css",
        "index.css",
    ]:

        path = os.path.join(
            PROJECT_ROOT,
            relative
        )

        if os.path.isfile(path):

            try:

                css = read_text(
                    path
                )

                css = re.sub(
                    r"@tailwind\s+[^;]+;",
                    "",
                    css
                )

                # Remove imports that would point to unavailable
                # local files.
                css = re.sub(
                    r"@import\s+['\"][^'\"]+['\"]\s*;",
                    "",
                    css
                )

                global_css += (
                    "\n/* " +
                    relative +
                    " */\n" +
                    css
                )

            except Exception:
                pass


    inspector = ""

    if inspect:

        inspector = inspector_script(
            channel
        )


    document = r"""
<!doctype html>

<html>

<head>

<meta charset="utf-8">

<meta
    name="viewport"
    content="width=device-width,initial-scale=1"
>

<title>Project Preview</title>


<script
    src="https://cdn.tailwindcss.com"
></script>


<style>

html,
body,
#root {
    min-height:100%;
}

body {
    margin:0;
}

__GLOBAL_CSS__

</style>

</head>


<body>

<div id="root"></div>


<script
    src="https://unpkg.com/react@18/umd/react.development.js"
></script>


<script
    src="https://unpkg.com/react-dom@18/umd/react-dom.development.js"
></script>


<script
    src="https://unpkg.com/@babel/standalone/babel.min.js"
></script>


<script>

(function(){

    var MODULES =
        __MODULES__;


    var ASSETS =
        __ASSETS__;


    var cache = {};


    function normalize(
        path
    ){

        var parts =
            path.split("/");

        var result = [];

        for(
            var i = 0;
            i < parts.length;
            i++
        ){

            var part =
                parts[i];

            if(
                !part ||
                part === "."
            ){
                continue;
            }

            if(part === ".."){

                if(result.length){
                    result.pop();
                }

                continue;
            }

            result.push(part);
        }

        return result.join("/");
    }


    function dirname(
        path
    ){

        var index =
            path.lastIndexOf("/");

        if(index < 0){
            return "";
        }

        return path.slice(
            0,
            index
        );
    }


    function withoutExtension(
        path
    ){

        return path.replace(
            /\.(tsx|jsx|ts|js|json|css|scss)$/i,
            ""
        );
    }


    function resolveLocal(
        request,
        from
    ){

        var base;


        if(
            request.startsWith("@/")
        ){

            base =
                request.slice(2);

        }else if(
            request.startsWith("~/")
        ){

            base =
                request.slice(2);

        }else if(
            request.startsWith("src/")
        ){

            base =
                request;

        }else if(
            request.startsWith("./")
            ||
            request.startsWith("../")
        ){

            base =
                normalize(
                    dirname(from) +
                    "/" +
                    request
                );

        }else{

            return null;
        }


        var candidates = [

            base,

            base + ".tsx",
            base + ".ts",
            base + ".jsx",
            base + ".js",

            base + ".json",

            base + ".css",
            base + ".scss",
            base + ".module.css",
            base + ".module.scss",

            base + "/index.tsx",
            base + "/index.ts",
            base + "/index.jsx",
            base + "/index.js"
        ];


        for(
            var i = 0;
            i < candidates.length;
            i++
        ){

            if(
                MODULES[
                    candidates[i]
                ] !== undefined
            ){

                return candidates[i];
            }
        }


        return null;
    }


    function findPublicAsset(
        request
    ){

        var value =
            String(request || "");


        value =
            value.split("?")[0];


        value =
            value.replace(
                /^\/+/,
                ""
            );


        if(
            ASSETS[value]
        ){

            return ASSETS[value];
        }


        var keys =
            Object.keys(
                ASSETS
            );


        for(
            var i = 0;
            i < keys.length;
            i++
        ){

            var key =
                keys[i];


            if(
                key.endsWith(value)
            ){

                return ASSETS[key];
            }
        }


        return null;
    }


    function genericIcon(
        name
    ){

        return React.forwardRef(
            function(
                props,
                ref
            ){

                var input =
                    Object.assign(
                        {},
                        props || {}
                    );

                var size =
                    input.size || 20;

                delete input.children;
                delete input.size;

                input.ref = ref;

                return React.createElement(
                    "svg",
                    Object.assign(
                        {},
                        input,
                        {
                            width:size,
                            height:size,
                            viewBox:"0 0 24 24",
                            fill:"none",
                            stroke:"currentColor",
                            strokeWidth:2,
                            strokeLinecap:"round",
                            strokeLinejoin:"round"
                        }
                    ),
                    React.createElement(
                        "circle",
                        {
                            cx:"12",
                            cy:"12",
                            r:"8"
                        }
                    )
                );
            }
        );
    }


    function genericComponent(
        name
    ){

        var lower =
            String(name).toLowerCase();

        var tag =
            "div";


        if(
            lower.includes("button")
            ||
            lower.includes("trigger")
            ||
            lower.includes("close")
            ||
            lower.includes("item")
        ){

            tag = "button";
        }


        if(lower === "input"){
            tag = "input";
        }


        if(lower === "textarea"){
            tag = "textarea";
        }


        if(lower === "select"){
            tag = "select";
        }


        if(lower.includes("label")){
            tag = "label";
        }


        if(lower.includes("image")){
            tag = "img";
        }


        if(lower.includes("separator")){
            tag = "hr";
        }


        return React.forwardRef(
            function(
                props,
                ref
            ){

                var input =
                    Object.assign(
                        {},
                        props || {}
                    );

                var children =
                    input.children;


                delete input.children;
                delete input.asChild;
                delete input.variant;
                delete input.size;
                delete input.onValueChange;


                if(
                    tag === "button"
                    &&
                    !input.type
                ){

                    input.type =
                        "button";
                }


                if(
                    tag === "img"
                    &&
                    !input.src
                ){

                    input.src =
                        "data:image/gif;base64,R0lGODlhAQABAAD/ACwAAAAAAQABAAACADs=";
                }


                input.ref = ref;


                return React.createElement(
                    tag,
                    input,
                    children
                );
            }
        );
    }


    function nextImage(
        props
    ){

        var input =
            Object.assign(
                {},
                props || {}
            );

        delete input.fill;
        delete input.priority;
        delete input.width;
        delete input.height;


        if(
            typeof input.src ===
            "string"
        ){

            var resolved =
                findPublicAsset(
                    input.src
                );

            if(resolved){
                input.src =
                    resolved;
            }
        }


        if(!input.src){

            input.src =
                "data:image/gif;base64,R0lGODlhAQABAAD/ACwAAAAAAQABAAACADs=";
        }


        return React.createElement(
            "img",
            input
        );
    }


    function nextFontProxy(){

        return new Proxy(
            {},
            {
                get:function(
                    target,
                    property
                ){

                    return function(){

                        return {
                            className:"",
                            variable:"",
                            style:{}
                        };

                    };
                }
            }
        );
    }


    function externalModule(
        request
    ){

        if(request === "react"){

            return React;
        }


        if(
            request === "react-dom"
            ||
            request === "react-dom/client"
        ){

            return ReactDOM;
        }


        if(
            request === "next/link"
        ){

            return {
                default:
                    genericComponent(
                        "Link"
                    )
            };
        }


        if(
            request === "next/image"
        ){

            return {
                default:
                    nextImage
            };
        }


        if(
            request === "next/navigation"
        ){

            return {

                useRouter:function(){

                    return {
                        push:function(url){
                            location.hash =
                                String(
                                    url || ""
                                );
                        },

                        replace:function(url){
                            location.hash =
                                String(
                                    url || ""
                                );
                        },

                        back:function(){
                            history.back();
                        },

                        refresh:function(){
                            location.reload();
                        },

                        prefetch:function(){}
                    };

                },


                usePathname:function(){
                    return location.pathname;
                },


                useSearchParams:function(){
                    return new URLSearchParams(
                        location.search
                    );
                }
            };
        }


        if(
            request.startsWith(
                "next/font/"
            )
        ){

            return nextFontProxy();
        }


        if(
            request === "lucide-react"
        ){

            return new Proxy(
                {},
                {
                    get:function(
                        target,
                        property
                    ){

                        return genericIcon(
                            String(property)
                        );

                    }
                }
            );
        }


        if(
            request === "framer-motion"
        ){

            var motion =
                new Proxy(
                    {},
                    {
                        get:function(
                            target,
                            property
                        ){

                            return genericComponent(
                                String(property)
                            );
                        }
                    }
                );


            return {
                motion:motion,

                AnimatePresence:function(
                    props
                ){

                    return React.createElement(
                        React.Fragment,
                        null,
                        props &&
                        props.children
                    );
                },

                useMotionValue:function(
                    value
                ){

                    return {
                        get:function(){
                            return value;
                        },

                        set:function(next){
                            value = next;
                        }
                    };
                }
            };
        }


        if(
            request === "clsx"
            ||
            request === "tailwind-merge"
        ){

            return function(){

                return Array
                    .from(arguments)
                    .flatMap(
                        function(value){

                            if(
                                typeof value ===
                                "string"
                            ){

                                return [value];
                            }

                            if(
                                Array.isArray(
                                    value
                                )
                            ){

                                return value;
                            }

                            if(
                                value &&
                                typeof value ===
                                "object"
                            ){

                                return Object.keys(
                                    value
                                ).filter(
                                    function(key){
                                        return Boolean(
                                            value[key]
                                        );
                                    }
                                );
                            }

                            return [];
                        }
                    )
                    .filter(Boolean)
                    .join(" ");
            };
        }


        if(
            request ===
            "class-variance-authority"
        ){

            return {
                cva:function(
                    base,
                    config
                ){

                    return function(
                        options
                    ){

                        return base || "";
                    };
                }
            };
        }


        if(
            request.startsWith(
                "@radix-ui/"
            )
        ){

            return new Proxy(
                {},
                {
                    get:function(
                        target,
                        property
                    ){

                        return genericComponent(
                            String(property)
                        );

                    }
                }
            );
        }


        // Unknown packages are represented by a lazy component
        // proxy. This keeps browser preview alive for visual UI
        // even when the real package runtime is unavailable.
        return new Proxy(
            {},
            {
                get:function(
                    target,
                    property
                ){

                    var name =
                        String(property);


                    if(
                        name.startsWith("use")
                    ){

                        return function(){
                            return undefined;
                        };
                    }


                    return genericComponent(
                        name
                    );
                }
            }
        );
    }


    function compileModule(
        path
    ){

        if(
            cache[path]
        ){

            return cache[path].exports;
        }


        var source =
            MODULES[path];


        if(
            source === undefined
        ){

            return {};
        }


        // CSS modules and styles are intentionally ignored.
        if(
            /\.(css|scss)$/i.test(path)
        ){

            cache[path] = {
                exports:{}
            };

            return {};
        }


        if(
            /\.json$/i.test(path)
        ){

            try{

                var parsed =
                    JSON.parse(
                        source
                    );

                cache[path] = {
                    exports:
                        parsed
                };

                return parsed;

            }catch(error){

                cache[path] = {
                    exports:{}
                };

                return {};
            }
        }


        var module = {
            exports:{}
        };


        cache[path] =
            module;


        function require(
            request
        ){

            // CSS
            if(
                /\.(css|scss)(\?[^?]*)?$/i.test(
                    request
                )
            ){

                return {};
            }


            // Image / asset import
            if(
                /\.(png|jpg|jpeg|webp|gif|svg|ico|avif)$/i.test(
                    request
                )
            ){

                var asset =
                    resolveLocalAsset(
                        request,
                        path
                    );

                return asset || {
                    src:asset || request
                };
            }


            var local =
                resolveLocal(
                    request,
                    path
                );


            if(local){

                return compileModule(
                    local
                );
            }


            return externalModule(
                request
            );
        }


        function resolveLocalAsset(
            request,
            from
        ){

            var base =
                request;


            if(
                request.startsWith("@/")
            ){

                base =
                    request.slice(2);

            }else if(
                request.startsWith("./")
                ||
                request.startsWith("../")
            ){

                base =
                    normalize(
                        dirname(from) +
                        "/" +
                        request
                    );
            }


            var direct =
                findPublicAsset(
                    base
                );


            if(direct){
                return direct;
            }


            var file =
                resolveLocal(
                    request,
                    from
                );


            if(file){

                var asset =
                    findPublicAsset(
                        file
                    );

                if(asset){
                    return asset;
                }
            }


            return null;
        }


        try{

            var compiled =
                Babel.transform(
                    source,
                    {
                        filename:path,
                        presets:[
                            "env",
                            "typescript",
                            "react"
                        ]
                    }
                ).code;


            var fn =
                new Function(
                    "require",
                    "module",
                    "exports",
                    compiled
                );


            fn(
                require,
                module,
                module.exports
            );


        }catch(error){

            console.error(
                "MODULE ERROR:",
                path,
                error
            );


            var box =
                document.createElement(
                    "pre"
                );


            box.style.cssText =
                "white-space:pre-wrap;" +
                "padding:14px;" +
                "margin:14px;" +
                "border-radius:8px;" +
                "background:#fee2e2;" +
                "color:#991b1b;" +
                "font:12px monospace;";


            box.textContent =
                "Module error: " +
                path +
                "\\n\\n" +
                (
                    error.stack ||
                    String(error)
                );


            document
                .getElementById(
                    "root"
                )
                .appendChild(
                    box
                );
        }


        return module.exports;
    }


    function showError(
        error
    ){

        var root =
            document.getElementById(
                "root"
            );


        root.innerHTML = "";


        var box =
            document.createElement(
                "pre"
            );


        box.style.cssText =
            "margin:20px;" +
            "padding:18px;" +
            "border-radius:10px;" +
            "background:#fee2e2;" +
            "color:#991b1b;" +
            "font:13px monospace;" +
            "white-space:pre-wrap;";


        box.textContent =
            "Preview Error\\n\\n" +
            (
                error.stack ||
                String(error)
            );


        root.appendChild(
            box
        );


        console.error(
            error
        );
    }


    try{

        var entryModule =
            compileModule(
                "__ENTRY__"
            );


        var App =
            entryModule.default ||
            entryModule.App ||
            entryModule.Page ||
            entryModule.Home ||
            entryModule;


        if(
            App &&
            App.default
        ){

            App =
                App.default;
        }


        if(
            typeof App !==
            "function"
            &&
            typeof App !==
            "object"
        ){

            throw new Error(
                "Could not resolve the project's root React component."
            );
        }


        ReactDOM
            .createRoot(
                document.getElementById(
                    "root"
                )
            )
            .render(
                React.createElement(
                    App
                )
            );


    }catch(error){

        showError(
            error
        );

    }

})();
</script>

__INSPECTOR__

</body>

</html>
"""

    document = document.replace(
        "__MODULES__",
        module_json
    )

    document = document.replace(
        "__ASSETS__",
        asset_json
    )

    document = document.replace(
        "__GLOBAL_CSS__",
        global_css
    )

    document = document.replace(
        "__ENTRY__",
        entry.replace(
            "\\",
            "/"
        )
    )

    document = document.replace(
        "__INSPECTOR__",
        inspector
    )

    return document


# ============================================================
# 11. RENDER PREVIEW
# ============================================================

def render_live_preview(
    inspect=False
):

    framework = (
        current_framework()
    )

    if not framework:

        display(
            HTML(
                """
                <div style="
                    padding:14px;
                    background:#fef3c7;
                    color:#92400e;
                    border-radius:9px;
                    font:13px system-ui;
                ">
                    ⚠️ Enter the framework / stack.
                </div>
                """
            )
        )

        return


    entry = find_entry()


    if not entry:

        display(
            HTML(
                """
                <div style="
                    padding:14px;
                    background:#fef3c7;
                    color:#92400e;
                    border-radius:9px;
                    font:13px system-ui;
                ">
                    ⚠️ No project entry file was detected.
                </div>
                """
            )
        )

        return


    channel = (
        "inspect-" +
        uuid.uuid4().hex
    )


    kind = framework_kind(
        framework
    )


    try:

        if (
            kind == "react"
            and entry.lower().endswith(
                (
                    ".tsx",
                    ".jsx",
                    ".ts",
                    ".js"
                )
            )
        ):

            page = make_react_preview(
                entry,
                inspect,
                channel
            )

            adapter = (
                "React / Next.js module preview"
            )

        elif entry.lower().endswith(
            (
                ".html",
                ".htm"
            )
        ):

            page = build_html_preview(
                read_text(
                    absolute_path(
                        entry
                    )
                ),
                inspect,
                channel
            )

            adapter = "HTML direct"

        else:

            display(
                HTML(
                    '<div style="'
                    'padding:14px;'
                    'background:#eff6ff;'
                    'color:#1e3a8a;'
                    'border-radius:9px;'
                    'font:13px system-ui;'
                    '">'
                    "Source editing works for "
                    + html.escape(framework)
                    + ".<br><br>"
                    "Live preview is currently implemented "
                    "for HTML and React / Next.js."
                    "</div>"
                )
            )

            return

    except Exception as exc:

        display(
            HTML(
                '<div style="'
                'padding:14px;'
                'background:#fee2e2;'
                'color:#991b1b;'
                'border-radius:9px;'
                'font:13px system-ui;'
                '">'
                "<b>❌ Preview construction failed</b>"
                "<br><br>"
                + html.escape(
                    str(exc)
                )
                + "</div>"
            )
        )

        return


    iframe_srcdoc = html.escape(
        page,
        quote=True
    )


    parent_script = """
<script>

(function(){

    var CHANNEL =
        __CHANNEL__;


    window.addEventListener(
        "message",
        function(event){

            var data =
                event.data || {};


            if(
                data.type !==
                "AI_INSPECTOR_SELECTION"
            ){
                return;
            }


            if(
                data.channel !==
                CHANNEL
            ){
                return;
            }


            var selector =
                document.querySelector(
                    ".refiner-selector-input input"
                );


            var details =
                document.querySelector(
                    ".refiner-details-input textarea"
                );


            if(selector){

                selector.value =
                    data.selector || "";


                selector.dispatchEvent(
                    new Event(
                        "input",
                        {
                            bubbles:true
                        }
                    )
                );


                selector.dispatchEvent(
                    new Event(
                        "change",
                        {
                            bubbles:true
                        }
                    )
                );
            }


            if(details){

                details.value =
                    data.details || "";


                details.dispatchEvent(
                    new Event(
                        "input",
                        {
                            bubbles:true
                        }
                    )
                );


                details.dispatchEvent(
                    new Event(
                        "change",
                        {
                            bubbles:true
                        }
                    )
                );
            }

        }
    );

})();

</script>
"""


    parent_script = parent_script.replace(
        "__CHANNEL__",
        json.dumps(channel)
    )


    mode_title = (
        "🔍 Inspect Mode"
        if inspect
        else
        "🌐 Live Preview"
    )


    preview = (
        '<div style="'
        'margin-top:15px;'
        'border:2px solid #cbd5e1;'
        'border-radius:10px;'
        'overflow:hidden;'
        'box-shadow:0 5px 16px rgba(0,0,0,.08);'
        '">'

        '<div style="'
        'padding:10px 13px;'
        'background:#e2e8f0;'
        'border-bottom:1px solid #cbd5e1;'
        'font:600 13px system-ui;'
        '">'

        + mode_title +

        '<span style="'
        'float:right;'
        'font-weight:400;'
        'opacity:.75;'
        '">'

        + html.escape(
            adapter
        ) +

        "</span>"
        "</div>"

        '<iframe '
        'srcdoc="'
        + iframe_srcdoc +
        '" '
        'style="'
        'display:block;'
        'width:100%;'
        'height:720px;'
        'border:0;'
        'background:white;'
        '" '
        'sandbox="allow-scripts allow-forms allow-modals allow-same-origin">'
        "</iframe>"

        "</div>"

        +
        parent_script
    )


    display(
        HTML(
            preview
        )
    )


# ============================================================
# 12. AI EDIT
# ============================================================

def call_devstral_edit(
    framework,
    instruction,
    selector,
    details,
    entry
):

    screenshot = (
        find_reference_screenshot()
    )

    screenshot_uri = (
        image_data_uri(
            screenshot
        )
        if screenshot
        else None
    )


    context = context_text(
        entry
    )


    tree = "\n".join(
        project_files()
    ) or "(empty project)"


    if selector:

        selected = (
            "SELECTED ELEMENT\n"
            "================\n"
            "Selector:\n"
            + selector +
            "\n\n"
            "Details:\n"
            + (
                details
                if details
                else "(none)"
            )
        )

    else:

        selected = (
            "No element is specifically selected."
        )


    prompt = (
        "You are a principal frontend engineer "
        "editing an EXISTING COMPLETE MULTI-FILE PROJECT.\n\n"

        "FRAMEWORK / STACK:\n"
        + framework +
        "\n\n"

        "ENTRY FILE:\n"
        + (
            entry
            if entry
            else "(none)"
        ) +
        "\n\n"

        "USER INSTRUCTION:\n"
        + instruction +
        "\n\n"

        + selected +

        "\n\nPROJECT TREE:\n"
        + tree +

        "\n\nRELEVANT PROJECT SOURCE:\n"
        + context +

        "\n\nSTRICT RULES:\n"
        "1. Keep the selected framework/stack.\n"
        "2. Do not migrate to another framework.\n"
        "3. Treat this as a complete project, not one component.\n"
        "4. Modify every file required to correctly implement the instruction.\n"
        "5. Create a new file when it is actually required.\n"
        "6. Update package.json only when a dependency change is actually required.\n"
        "7. Delete a file only when the instruction actually requires it.\n"
        "8. Preserve all unrelated files and functionality.\n"
        "9. Preserve existing content unless the instruction changes it.\n"
        "10. Keep imports valid.\n"
        "11. Keep paths valid.\n"
        "12. Do not invent broken imports.\n"
        "13. Do not use placeholders.\n"
        "14. Do not use '...'.\n"
        "15. Do not return a diff.\n"
        "16. Do not return explanations outside JSON.\n\n"

        "RETURN VALID JSON ONLY:\n"
        "{\n"
        '  "summary": "description",\n'
        '  "files": [\n'
        "    {\n"
        '      "action": "update",\n'
        '      "path": "app/page.tsx",\n'
        '      "content": "COMPLETE FILE CONTENT"\n'
        "    },\n"
        "    {\n"
        '      "action": "create",\n'
        '      "path": "components/NewThing.tsx",\n'
        '      "content": "COMPLETE FILE CONTENT"\n'
        "    },\n"
        "    {\n"
        '      "action": "delete",\n'
        '      "path": "old/file.tsx"\n'
        "    }\n"
        "  ]\n"
        "}\n"
    )


    messages = [
        {
            "role":
                "system",

            "content":
                "You are Devstral Small 2 acting as "
                "a precise multi-file project editor. "
                "Return valid JSON only."
        }
    ]


    user_content = [
        {
            "type":
                "text",

            "text":
                prompt
        }
    ]


    if screenshot_uri:

        user_content.append(
            {
                "type":
                    "image_url",

                "image_url":
                    {
                        "url":
                            screenshot_uri
                    }
            }
        )


    messages.append(
        {
            "role":
                "user",

            "content":
                user_content
        }
    )


    response = requests.post(
        LOCAL_API +
        "/chat/completions",

        json={
            "model":
                LOCAL_MODEL,

            "messages":
                messages,

            "temperature":
                0.08,

            "top_p":
                0.90,

            "max_tokens":
                14000,

            "stream":
                False
        },

        timeout=1200
    )


    response.raise_for_status()


    result = response.json()


    if "choices" not in result:

        raise RuntimeError(
            "Invalid Devstral response:\n" +
            json.dumps(
                result,
                indent=2
            )[:12000]
        )


    content = (
        result["choices"][0]
        ["message"]
        ["content"]
    )


    if isinstance(
        content,
        list
    ):

        parts = []

        for item in content:

            if isinstance(
                item,
                dict
            ):

                if item.get(
                    "type"
                ) == "text":

                    parts.append(
                        item.get(
                            "text",
                            ""
                        )
                    )

            elif isinstance(
                item,
                str
            ):

                parts.append(
                    item
                )

        content = "\n".join(
            parts
        )


    text = str(
        content
    ).strip()


    fenced = re.search(
        r"```(?:json)?\s*([\s\S]*?)\s*```",
        text,
        re.IGNORECASE
    )


    if fenced:

        text = fenced.group(
            1
        ).strip()


    first = text.find("{")
    last = text.rfind("}")


    if first >= 0 and last > first:

        text = text[
            first:last + 1
        ]


    manifest = json.loads(
        text
    )


    changes = manifest.get(
        "files",
        []
    )


    if not isinstance(
        changes,
        list
    ):

        raise ValueError(
            "AI manifest has no valid files list."
        )


    clean = []

    for item in changes:

        if not isinstance(
            item,
            dict
        ):
            continue

        action = str(
            item.get(
                "action",
                "update"
            )
        ).lower().strip()

        path = item.get(
            "path"
        )

        if action not in {
            "update",
            "create",
            "delete"
        }:
            continue

        if not path:
            continue

        safe = str(
            path
        ).replace(
            "\\",
            "/"
        ).lstrip("/")
        safe = "/".join(
            x for x in safe.split("/")
            if x not in {
                "",
                ".",
                ".."
            }
        )

        if action != "delete":
            content = item.get(
                "content"
            )

            if content is None:
                raise ValueError(
                    "Missing content for: " +
                    safe
                )

            clean.append(
                {
                    "action":
                        action,

                    "path":
                        safe,

                    "content":
                        str(content)
                }
            )

        else:

            clean.append(
                {
                    "action":
                        action,

                    "path":
                        safe
                }
            )


    if not clean:

        raise ValueError(
            "AI returned no usable project changes."
        )


    return (
        clean,
        manifest
    )


# ============================================================
# 13. APPLY CHANGES
# ============================================================

def apply_changes(
    changes
):

    previous = {}
    backups = []

    try:

        # Validate existing files first.
        for item in changes:

            action = item["action"]
            path = item["path"]

            absolute = absolute_path(
                path
            )

            if action in {
                "update",
                "delete"
            }:

                if not os.path.isfile(
                    absolute
                ):

                    raise ValueError(
                        action.upper() +
                        " target does not exist: " +
                        path
                    )

                previous[path] = (
                    read_text(
                        absolute
                    )
                )

        # Backup
        for path in previous:

            backup_path = backup(
                absolute_path(path)
            )

            if backup_path:
                backups.append(
                    backup_path
                )

        created = []
        updated = []
        deleted = []

        for item in changes:

            action = item["action"]
            path = item["path"]
            absolute = absolute_path(path)

            if action == "delete":

                os.remove(
                    absolute
                )

                deleted.append(
                    path
                )

                continue

            atomic_write(
                absolute,
                item["content"]
            )

            if path in previous:

                updated.append(
                    path
                )

            else:

                created.append(
                    path
                )


        return {
            "created": created,
            "updated": updated,
            "deleted": deleted,
            "backups": backups
        }


    except Exception:

        # Restore old files.
        for path, content in previous.items():

            try:

                atomic_write(
                    absolute_path(path),
                    content
                )

            except Exception:
                pass

        # Remove files that were created during this edit.
        for item in changes:

            if item["action"] != "create":
                continue

            path = item["path"]

            if path in previous:
                continue

            try:

                absolute = absolute_path(
                    path
                )

                if os.path.exists(
                    absolute
                ):

                    os.remove(
                        absolute
                    )

            except Exception:
                pass

        raise


# ============================================================
# 14. ZIP
# ============================================================

def export_zip():

    path = (
        "/kaggle/working/"
        "screenshot-ui-refined-project.zip"
    )

    if os.path.exists(path):
        os.remove(path)

    with zipfile.ZipFile(
        path,
        "w",
        zipfile.ZIP_DEFLATED
    ) as archive:

        for root, dirs, files in os.walk(
            PROJECT_ROOT
        ):

            dirs[:] = [
                d for d in dirs
                if d not in {
                    ".git",
                    "node_modules",
                    "__pycache__",
                    ".next",
                    ".preview",
                }
            ]

            for name in files:

                full = os.path.join(
                    root,
                    name
                )

                if any(
                    x in full.lower()
                    for x in [
                        ".backup_",
                        ".bak",
                        ".tmp"
                    ]
                ):
                    continue

                archive.write(
                    full,
                    os.path.relpath(
                        full,
                        PROJECT_ROOT
                    ).replace(
                        "\\",
                        "/"
                    )
                )

    return path


# ============================================================
# 15. CALLBACKS
# ============================================================

_running = False


def update_status(
    _=None
):

    framework = current_framework()

    if not framework:

        source_status.value = """
        <div style="
            padding:10px;
            background:#fffbeb;
            color:#92400e;
            border:1px solid #f59e0b;
            border-radius:7px;
            font:13px system-ui;
        ">
            ⚠️ Enter the framework / stack.
        </div>
        """

        return


    files = project_files()
    entry = find_entry()


    if not files:

        source_status.value = (
            '<div style="'
            'padding:10px;'
            'background:#fffbeb;'
            'color:#92400e;'
            'border:1px solid #f59e0b;'
            'border-radius:7px;'
            'font:13px system-ui;'
            '">'
            "⚠️ Project is empty."
            "</div>"
        )

        return


    source_status.value = (
        '<div style="'
        'padding:10px 14px;'
        'background:#f0fdf4;'
        'color:#166534;'
        'border:1px solid #86efac;'
        'border-radius:7px;'
        'font:13px system-ui;'
        '">'
        "✅ <b>Framework:</b> " +
        html.escape(
            framework
        ) +
        "<br>"
        "📁 <b>Project:</b> " +
        html.escape(
            PROJECT_ROOT
        ) +
        "<br>"
        "📦 <b>Files:</b> " +
        str(
            len(files)
        ) +
        "<br>"
        "🚪 <b>Entry:</b> " +
        (
            html.escape(entry)
            if entry
            else "Not found"
        )
        + "</div>"
    )


def on_preview(
    _
):

    with refine_output:

        clear_output()

        print(
            "=" * 72
        )

        print(
            "🌐 LIVE PREVIEW"
        )

        print(
            "=" * 72
        )

        print(
            "Framework:",
            current_framework()
        )

        print(
            "Entry:",
            find_entry()
        )

        print()

        render_live_preview(
            False
        )


def on_inspect(
    _
):

    with refine_output:

        clear_output()

        print(
            "=" * 72
        )

        print(
            "🔍 INSPECT & EDIT"
        )

        print(
            "=" * 72
        )

        print(
            "\n👉 Move your mouse over the preview."
        )

        print(
            "👉 Click the exact element to edit."
        )

        print(
            "👉 Selector and element details will "
            "appear above."
        )

        print()

        render_live_preview(
            True
        )


def clear_selection(
    _
):

    selected_selector.value = ""
    selected_details.value = ""

    with refine_output:

        clear_output()

        print(
            "✅ Selection cleared."
        )


def on_refine(
    _
):

    global _running

    if _running:
        return

    framework = current_framework()

    instruction = (
        edit_instruction
        .value
        .strip()
    )

    selector = (
        selected_selector
        .value
        .strip()
    )

    details = (
        selected_details
        .value
        .strip()
    )

    entry = find_entry()


    if not framework:

        with refine_output:

            clear_output()

            print(
                "⚠️ Enter the framework / stack."
            )

        return


    if not instruction:

        with refine_output:

            clear_output()

            print(
                "⚠️ Enter the edit instruction."
            )

        return


    if not entry:

        with refine_output:

            clear_output()

            print(
                "❌ Could not find project entry file."
            )

        return


    _running = True


    refine_button.disabled = True
    preview_button.disabled = True
    inspect_button.disabled = True
    clear_button.disabled = True


    old_label = (
        refine_button.description
    )

    refine_button.description = (
        "⏳ Devstral Editing..."
    )


    try:

        with refine_output:

            clear_output()

            print(
                "=" * 72
            )

            print(
                "✨ COMPLETE PROJECT REFINER"
            )

            print(
                "=" * 72
            )

            print(
                "\n🧠 Model:",
                LOCAL_MODEL
            )

            print(
                "🔗 API:",
                LOCAL_API
            )

            print(
                "🧩 Framework:",
                framework
            )

            print(
                "📄 Entry:",
                entry
            )

            print(
                "📦 Current files:",
                len(
                    project_files()
                )
            )

            print(
                "\n🎯 Selector:",
                selector or "None"
            )

            print(
                "\n✏️ Instruction:"
            )

            print(
                instruction
            )


            # Check model
            requests.get(
                LOCAL_API +
                "/models",
                timeout=8
            ).raise_for_status()


            print(
                "\n✅ Local Devstral reachable."
            )


            print(
                "\n🤖 Devstral is analyzing the complete project..."
            )


            started = time.time()


            changes, manifest = (
                call_devstral_edit(
                    framework,
                    instruction,
                    selector,
                    details,
                    entry
                )
            )


            elapsed = (
                time.time() -
                started
            )


            print(
                f"⏱️ AI time: {elapsed:.1f}s"
            )


            print(
                "\n📝 Summary:"
            )

            print(
                manifest.get(
                    "summary",
                    "Project updated."
                )
            )


            print(
                "\n📂 Changes:"
            )

            for item in changes:

                print(
                    "  " +
                    item["action"].upper() +
                    "  " +
                    item["path"]
                )


            result = apply_changes(
                changes
            )


            print(
                "\n✅ Project updated successfully."
            )


            if result["created"]:

                print(
                    "\n🆕 Created:"
                )

                for path in result["created"]:
                    print(
                        "  + " + path
                    )


            if result["updated"]:

                print(
                    "\n✏️ Updated:"
                )

                for path in result["updated"]:
                    print(
                        "  ~ " + path
                    )


            if result["deleted"]:

                print(
                    "\n🗑️ Deleted:"
                )

                for path in result["deleted"]:
                    print(
                        "  - " + path
                    )


            if result["backups"]:

                print(
                    "\n💾 Backups:",
                    len(
                        result["backups"]
                    )
                )


            edit_instruction.value = ""

            selected_selector.value = ""
            selected_details.value = ""


            update_status()


            zip_path = export_zip()


            print(
                "\n📦 Updated complete project ZIP:"
            )


            display(
                FileLink(
                    zip_path,
                    result_html_prefix=(
                        "⬇️ Download Complete Project: "
                    )
                )
            )


            print(
                "\n🌐 Refreshing preview..."
            )


            render_live_preview(
                False
            )


    except Exception as exc:

        with refine_output:

            print(
                "\n❌ Refinement failed:"
            )

            print(
                repr(exc)
            )


    finally:

        _running = False

        refine_button.disabled = False
        preview_button.disabled = False
        inspect_button.disabled = False
        clear_button.disabled = False

        refine_button.description = (
            old_label
        )


# ============================================================
# 16. WIDGETS
# ============================================================

refine_header = widgets.HTML(
    """
    <div style="
        margin:20px 0 14px;
        padding:19px;
        border-radius:11px;
        background:linear-gradient(
            135deg,
            #111827,
            #1d4ed8
        );
        color:white;
    ">

        <h2 style="
            margin:0 0 6px;
        ">
            ✨ Complete Project AI Refiner
        </h2>

        <div style="
            font-size:13px;
            opacity:.92;
        ">
            Multi-file editing · Working React/Next preview ·
            Inspect & Edit · Local Devstral Small 2
        </div>

    </div>
    """
)


refine_framework_input = widgets.Text(
    value=previous_framework(),

    description="Framework / Stack:",

    placeholder=(
        "Next.js + TypeScript + Tailwind CSS + shadcn/ui"
    ),

    layout=widgets.Layout(
        width="850px"
    ),

    style={
        "description_width":
            "initial"
    }
)


framework_help = widgets.HTML(
    """
    <div style="
        margin-bottom:14px;
        padding:11px 14px;
        border-left:4px solid #2563eb;
        background:#eff6ff;
        border-radius:7px;
        font:13px system-ui;
    ">

        <b>ONE exact framework / stack.</b>

        The refiner edits the COMPLETE PROJECT,
        not only ScreenshotUI.jsx.

        <br><br>

        Example:
        <code>
        Next.js + TypeScript + Tailwind CSS + shadcn/ui
        </code>

    </div>
    """
)


model_status = widgets.HTML(
    '<div style="'
    'padding:11px 14px;'
    'margin-bottom:14px;'
    'border:1px solid #d1d5db;'
    'border-radius:7px;'
    'background:#f9fafb;'
    'font:13px system-ui;'
    '">'
    "🧠 <b>Local model:</b> " +
    html.escape(
        LOCAL_MODEL
    ) +
    "<br>"
    "🔗 <b>API:</b> " +
    html.escape(
        LOCAL_API
    ) +
    "</div>"
)


source_status = widgets.HTML("")


selected_selector = widgets.Text(
    value="",

    description="Selector:",

    placeholder=(
        "Click an element in Inspect & Edit"
    ),

    layout=widgets.Layout(
        width="700px"
    ),

    style={
        "description_width":
            "initial"
    }
)


selected_details = widgets.Textarea(
    value="",

    description="Element:",

    placeholder=(
        "Selected element details appear here."
    ),

    layout=widgets.Layout(
        width="700px",
        height="145px"
    ),

    style={
        "description_width":
            "initial"
    }
)


edit_instruction = widgets.Textarea(
    value="",

    description="Instruction:",

    placeholder=(
        "Example:\n"
        "Change this button to blue.\n"
        "Add a collapsible sidebar.\n"
        "Make the dashboard responsive.\n"
        "Create a reusable card component."
    ),

    layout=widgets.Layout(
        width="850px",
        height="135px"
    ),

    style={
        "description_width":
            "initial"
    }
)


refine_button = widgets.Button(
    description="✨ Apply AI Edit",

    button_style="warning",

    layout=widgets.Layout(
        width="180px",
        height="44px"
    )
)


preview_button = widgets.Button(
    description="🌐 Live Preview",

    button_style="info",

    layout=widgets.Layout(
        width="170px",
        height="44px"
    )
)


inspect_button = widgets.Button(
    description="🔍 Inspect & Edit",

    button_style="success",

    layout=widgets.Layout(
        width="185px",
        height="44px"
    )
)


clear_button = widgets.Button(
    description="✕ Clear Selection",

    layout=widgets.Layout(
        width="160px",
        height="34px"
    )
)


refine_output = widgets.Output()


# ============================================================
# 17. CLASS MARKERS
# ============================================================

selected_selector.add_class(
    "refiner-selector-input"
)

selected_details.add_class(
    "refiner-details-input"
)


# ============================================================
# 18. EVENTS
# ============================================================

refine_framework_input.observe(
    update_status,
    names="value"
)

refine_button.on_click(
    on_refine
)

preview_button.on_click(
    on_preview
)

inspect_button.on_click(
    on_inspect
)

clear_button.on_click(
    clear_selection
)


# ============================================================
# 19. DISPLAY
# ============================================================

display(
    refine_header
)

display(
    refine_framework_input
)

display(
    framework_help
)

display(
    model_status
)

display(
    source_status
)

display(
    widgets.HTML(
        "<b>🎯 Selected Element</b>"
    )
)

display(
    selected_selector
)

display(
    selected_details
)

display(
    clear_button
)

display(
    widgets.HTML(
        """
        <div style="
            font-weight:700;
            margin-top:14px;
            margin-bottom:7px;
        ">
            ✏️ Project Edit Instruction
        </div>
        """
    )
)

display(
    edit_instruction
)

display(
    widgets.HBox(
        [
            refine_button,
            preview_button,
            inspect_button
        ],
        layout=widgets.Layout(
            gap="10px"
        )
    )
)

display(
    refine_output
)


# ============================================================
# 20. READY
# ============================================================

update_status()

print(
    "✅ CELL 11 loaded successfully."
)

print(
    "🧠 Local model:",
    LOCAL_MODEL
)

print(
    "🔗 API:",
    LOCAL_API
)

print(
    "📁 Project:",
    PROJECT_ROOT
)

print(
    "\nWorkflow:"
)

print(
    "COMPLETE PROJECT"
    " → Live Preview"
    " → Inspect"
    " → Select Element"
    " → Instruction"
    " → Devstral"
    " → CREATE / UPDATE / DELETE files"
    " → Backup"
    " → ZIP"
)


HTML(value='\n    <div style="\n        margin:20px 0 14px;\n        padding:19px;\n        border-radius:11px…

Text(value='Next.js + TypeScript + Tailwind CSS + shadcn/ui', description='Framework / Stack:', layout=Layout(…

HTML(value='\n    <div style="\n        margin-bottom:14px;\n        padding:11px 14px;\n        border-left:4…

HTML(value='<div style="padding:11px 14px;margin-bottom:14px;border:1px solid #d1d5db;border-radius:7px;backgr…

HTML(value='')

...

Button(description='✕ Clear Selection', layout=Layout(height='34px', width='160px'), style=ButtonStyle())

HTML(value='\n        <div style="\n            font-weight:700;\n            margin-top:14px;\n            ma…

Textarea(value='', description='Instruction:', layout=Layout(height='135px', width='850px'), placeholder='Exam…

Output()

✅ CELL 11 loaded successfully.
🧠 Local model: mistralai/Devstral-Small-2-24B-Instruct-2512
🔗 API: http://127.0.0.1:8000/v1
📁 Project: /kaggle/working/generated_project

Workflow:
COMPLETE PROJECT → Live Preview → Inspect → Select Element → Instruction → Devstral → CREATE / UPDATE / DELETE files → Backup → ZIP


In [59]:
# ============================================================
# KAGGLE FILE MANAGER + TERMINAL
# ============================================================

import os
import subprocess
import shlex
import html
import ipywidgets as widgets

from IPython.display import display, HTML, clear_output, FileLink


# ============================================================
# CONFIG
# ============================================================

START_DIR = "/kaggle/working"

if not os.path.exists(START_DIR):
    os.makedirs(START_DIR, exist_ok=True)


current_dir = START_DIR


# ============================================================
# HELPERS
# ============================================================

def safe_path(path):
    path = os.path.abspath(path)

    if not (
        path == "/kaggle"
        or path.startswith("/kaggle/")
    ):
        raise ValueError(
            "Access restricted to /kaggle"
        )

    return path


def format_size(size):

    units = [
        "B",
        "KB",
        "MB",
        "GB",
        "TB"
    ]

    value = float(size)

    for unit in units:

        if value < 1024:
            return f"{value:.1f} {unit}"

        value /= 1024

    return f"{value:.1f} PB"


# ============================================================
# FILE MANAGER
# ============================================================

file_output = widgets.Output()


path_input = widgets.Text(
    value=START_DIR,
    description="Path:",
    layout=widgets.Layout(
        width="800px"
    ),
    style={
        "description_width": "70px"
    }
)


refresh_button = widgets.Button(
    description="🔄 Refresh",
    button_style="info",
    layout=widgets.Layout(
        width="120px"
    )
)


up_button = widgets.Button(
    description="⬆️ Up",
    layout=widgets.Layout(
        width="90px"
    )
)


def show_files(path=None):

    global current_dir

    if path is None:
        path = path_input.value.strip()

    try:

        path = safe_path(path)

        if not os.path.isdir(path):
            raise ValueError(
                "Not a directory: " + path
            )

        current_dir = path

        path_input.value = path

    except Exception as exc:

        with file_output:

            clear_output()

            print(
                "❌ " + str(exc)
            )

        return


    with file_output:

        clear_output()

        print(
            f"📁 {current_dir}"
        )

        print(
            "-" * 80
        )


        try:

            entries = sorted(
                os.listdir(
                    current_dir
                ),
                key=lambda x: (
                    not os.path.isdir(
                        os.path.join(
                            current_dir,
                            x
                        )
                    ),
                    x.lower()
                )
            )

        except Exception as exc:

            print(
                "❌ Cannot read directory:",
                exc
            )

            return


        if not entries:

            print(
                "(empty directory)"
            )

            return


        for name in entries:

            if name.startswith(
                "."
            ):
                continue

            full_path = os.path.join(
                current_dir,
                name
            )


            # ------------------------------------------------
            # DIRECTORY
            # ------------------------------------------------

            if os.path.isdir(full_path):

                button = widgets.Button(
                    description="📁 " + name,
                    layout=widgets.Layout(
                        width="500px"
                    )
                )


                def open_folder(
                    _,
                    folder=full_path
                ):

                    show_files(
                        folder
                    )


                button.on_click(
                    open_folder
                )

                display(
                    button
                )

                continue


            # ------------------------------------------------
            # FILE
            # ------------------------------------------------

            try:

                size = os.path.getsize(
                    full_path
                )

            except Exception:

                size = 0


            row = widgets.HBox(
                [
                    widgets.HTML(
                        "<div style='"
                        "width:420px;"
                        "padding:6px 0;"
                        "font-family:monospace;"
                        "'>"
                        + html.escape(name)
                        + "</div>"
                    ),

                    widgets.HTML(
                        "<div style='"
                        "width:100px;"
                        "padding:6px 0;"
                        "color:#6b7280;"
                        "'>"
                        + format_size(size)
                        + "</div>"
                    )
                ]
            )


            display(
                row
            )


            # Download link
            try:

                display(
                    FileLink(
                        full_path,
                        result_html_prefix="⬇️ "
                    )
                )

            except Exception:
                pass


# ============================================================
# PATH EVENTS
# ============================================================

def refresh_clicked(_):

    show_files(
        path_input.value
    )


def up_clicked(_):

    parent = os.path.dirname(
        current_dir.rstrip("/")
    )

    if not parent:
        parent = "/kaggle"

    show_files(
        parent
    )


refresh_button.on_click(
    refresh_clicked
)

up_button.on_click(
    up_clicked
)


# ============================================================
# TERMINAL
# ============================================================

terminal_output = widgets.Output()


command_input = widgets.Text(
    value="",

    placeholder=(
        "Example: ls -lah /kaggle/working"
    ),

    description="$",

    layout=widgets.Layout(
        width="800px"
    ),

    style={
        "description_width": "20px"
    }
)


run_button = widgets.Button(
    description="▶ Run",
    button_style="success",
    layout=widgets.Layout(
        width="100px"
    )
)


clear_terminal_button = widgets.Button(
    description="🗑️ Clear",
    layout=widgets.Layout(
        width="110px"
    )
)


def run_terminal(
    _
):

    command = (
        command_input.value
        .strip()
    )

    if not command:

        return


    with terminal_output:

        print(
            f"\n$ {command}"
        )


        try:

            process = subprocess.run(
                command,
                shell=True,
                executable="/bin/bash",

                cwd=current_dir,

                stdout=subprocess.PIPE,

                stderr=subprocess.STDOUT,

                text=True,

                timeout=600
            )


            output = process.stdout or ""


            if output:

                print(
                    output
                )


            if process.returncode == 0:

                print(
                    f"✅ Exit code: "
                    f"{process.returncode}"
                )

            else:

                print(
                    f"❌ Exit code: "
                    f"{process.returncode}"
                )


        except subprocess.TimeoutExpired:

            print(
                "❌ Command timed out."
            )


        except Exception as exc:

            print(
                "❌ Terminal error:",
                repr(exc)
            )


    command_input.value = ""


def clear_terminal(
    _
):

    with terminal_output:

        clear_output()


run_button.on_click(
    run_terminal
)

clear_terminal_button.on_click(
    clear_terminal
)


# ============================================================
# QUICK TERMINAL COMMANDS
# ============================================================

def quick_command(
    command
):

    def handler(_):

        command_input.value = command

        run_terminal(None)

    return handler


quick_commands = widgets.HBox(
    [
        widgets.Button(
            description="ls",
            layout=widgets.Layout(
                width="70px"
            )
        ),

        widgets.Button(
            description="pwd",
            layout=widgets.Layout(
                width="70px"
            )
        ),

        widgets.Button(
            description="df -h",
            layout=widgets.Layout(
                width="90px"
            )
        ),

        widgets.Button(
            description="nvidia-smi",
            layout=widgets.Layout(
                width="110px"
            )
        ),

        widgets.Button(
            description="ps",
            layout=widgets.Layout(
                width="70px"
            )
        )
    ]
)


quick_buttons = list(
    quick_commands.children
)

quick_buttons[0].on_click(
    quick_command(
        "ls -lah"
    )
)

quick_buttons[1].on_click(
    quick_command(
        "pwd"
    )
)

quick_buttons[2].on_click(
    quick_command(
        "df -h"
    )
)

quick_buttons[3].on_click(
    quick_command(
        "nvidia-smi"
    )
)

quick_buttons[4].on_click(
    quick_command(
        "ps aux --sort=-%mem | head -20"
    )
)


# ============================================================
# UI
# ============================================================

header = widgets.HTML(
    """
    <div style="
        background:linear-gradient(
            135deg,
            #111827,
            #1d4ed8
        );
        color:white;
        padding:18px;
        border-radius:10px;
        margin-bottom:15px;
    ">

        <h2 style="
            margin:0 0 5px;
        ">
            🗂️ Kaggle File Manager + Terminal
        </h2>

        <div style="
            font-size:13px;
            opacity:.9;
        ">
            Browse files, download files, and run shell commands
            inside the Kaggle notebook environment.
        </div>

    </div>
    """
)


display(
    header
)


# File manager

display(
    widgets.HTML(
        "<h3>🗂️ File Manager</h3>"
    )
)


display(
    widgets.HBox(
        [
            path_input,
            refresh_button,
            up_button
        ]
    )
)


display(
    file_output
)


# Terminal

display(
    widgets.HTML(
        "<h3 style='margin-top:25px;'>"
        "💻 Terminal"
        "</h3>"
    )
)


display(
    widgets.HBox(
        [
            command_input,
            run_button,
            clear_terminal_button
        ]
    )
)


display(
    widgets.HTML(
        "<div style='"
        "font-size:12px;"
        "color:#6b7280;"
        "margin:5px 0 8px;"
        "'>"
        "Working directory follows the File Manager path."
        "</div>"
    )
)


display(
    quick_commands
)


display(
    terminal_output
)


# Initial file listing

show_files(
    START_DIR
)


print(
    "\n✅ File Manager + Terminal ready."
)

print(
    "📁 Root:",
    START_DIR
)

HTML(value='\n    <div style="\n        background:linear-gradient(\n            135deg,\n            #111827,…

HTML(value='<h3>🗂️ File Manager</h3>')

Output()

HTML(value="<h3 style='margin-top:25px;'>💻 Terminal</h3>")

HTML(value="<div style='font-size:12px;color:#6b7280;margin:5px 0 8px;'>Working directory follows the File Man…

Output()


✅ File Manager + Terminal ready.
📁 Root: /kaggle/working
